<a href="https://colab.research.google.com/github/finneKIM/stem-remix-assistant/blob/main/notebooks/02a_regenerate_musicongen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02a — EXP-002: MusiConGen 재생성 파이프라인

Demucs로 분리한 원곡 스템 중 하나(`target_stem`)를 MusiConGen으로 재생성하고, 다시 Demucs로 분리한 뒤 madmom 기반 Alignment Engine으로 원곡과 타이밍을 맞추는 파이프라인.

설정값은 `experiments/exp_002_musicongen/config.yaml` 참고. 파이프라인 전체 구조는 Notion "02번 노트북 — 재생성/재조합 로직 설계" 페이지.

**실행 환경**: Google Colab, GPU 런타임(T4) 필수. MusiConGen 추론에 12GB+ VRAM 권장, T4는 16GB라 충족.

**실행의 목적**: `duration_sec`를 최소(10s)/중간(15s)/최대(20s) 세 구간으로 스윕, `seed`는 42로 전 구간 고정해서 최적 duration_sec 탐색 (2026-08-28 TODO).

## 0. GPU 확인

In [ ]:
!nvidia-smi|

/bin/bash: -c: line 2: syntax error: unexpected end of file


In [ ]:
!ls -la /content/

total 24
drwxr-xr-x 1 root root 4096 Sep 14 07:17 .
drwxr-xr-x 1 root root 4096 Sep 14 07:11 ..
drwxr-xr-x 4 root root 4096 Sep  4 13:32 .config
drwxr-xr-x 5 root root 4096 Sep 14 07:17 MusiConGen
-rw-r--r-- 1 root root    0 Sep 14 07:17 requirements_no_xformers.txt
drwxr-xr-x 1 root root 4096 Sep  4 13:32 sample_data
drwxr-xr-x 6 root root 4096 Sep 14 07:16 stem-remix-assistant


## 1. 저장소 클론 + MusiConGen 설치

- `stem-remix-assistant`: 원곡/스템 샘플(`docs/samples/`)과 `config.yaml` 확보용
- `MusiConGen`: 공식 저장소 (https://github.com/YatingMusic/MusiConGen)

In [ ]:
!git clone https://github.com/finneKIM/stem-remix-assistant.git
!git clone https://github.com/YatingMusic/MusiConGen.git

%cd MusiConGen
!pip install -r requirements.txt -q
!conda install -y 'ffmpeg<5' -c conda-forge 2>/dev/null || apt-get -y install ffmpeg -q


fatal: destination path 'stem-remix-assistant' already exists and is not an empty directory.
fatal: destination path 'MusiConGen' already exists and is not an empty directory.
/content/MusiConGen
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.


In [ ]:
!python --version

Python 3.13.15


#### 실험 - requirements.txt에서 xforemrs 줄 제외하고 설치
- requirements.txt를 그대로 쓰지 않고 xformers 줄만 걸러서 설치
- MusiConGen/audiocraft가 xFormers 없이도 정상 임포트되는지 확인
- 실제 생성 확인으로 xFormers 없이도 파이프라인이 도는지 검증

In [ ]:
# reuiqrements.txt에서 xFormers 줄만 제외하고 설치
!grep -v -i "xforemrs" requirements.txt > requirements_no_xformers.txt
!cat requirements_no_xformers.txt

av==11.0.0
einops
flashy==0.0.1
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
torch==2.0.0
torchaudio==2.0.0
tqdm
transformers==4.31.0  # need Encodec there.
xformers==0.0.22
demucs
librosa
soundfile
torchmetrics
encodec
protobuf
torchvision==0.16.0
torchtext==0.16.0
pesq
pystoi


In [ ]:
# torch 버전과 python 3.13 버전 호환여부 확인
!pip index versions torch 2>&1 | head -5

torch (2.14.0)
Available versions: 2.14.0, 2.13.0, 2.12.1, 2.12.0, 2.11.0, 2.10.0, 2.9.1, 2.9.0, 2.8.0, 2.7.1, 2.7.0, 2.6.0, 2.5.1, 2.5.0
  INSTALLED: 2.11.0+cu128
  LATEST:    2.14.0


In [ ]:
import torch, torchaudio, torchvision, torchtext
print("torch:", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("torchvision:", torchvision.__version__)
print("torchtext:", torchtext.__version__)

ModuleNotFoundError: No module named 'torchtext'

In [ ]:
# torch, torchaudio, torchvision, torchtext를 requirements.txt에서 제외
# colab 환경의 torch 2.11.0 환경 그대로 나머지 패키지 설치
# import audiocraft가 실제로 torchtext를 요구하는지 에러로 직접 확인

!grep -v -iE "^(xformers|torch|torchaudio|torchvision|torchtext)(==|>=|<)?" requirements.txt > requirements_filtered.txt
!cat requirements_filtered.txt

av==11.0.0
einops
flashy==0.0.1
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [ ]:
# numpy 버전 확인 및 그레이드 충돌 or python3.13 wheel 부재 문제 사전확인
# 설치 충동 에러 발생 사전 확인
!pip install -r requirements_filtered.txt

  Using cached av-11.0.0.tar.gz (3.7 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
!cat requirements_filtered.txt

av==11.0.0
einops
flashy==0.0.1
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [ ]:
# av 버전 a안 실행
!sed -i 's/^av==11.0.0/av/' requirements_filtered.txt
!cat requirements_filtered.txt

av
einops
flashy==0.0.1
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [ ]:
!pip install -r requirements_filtered.txt -v 2>&1 | tail -100

    Found link https://files.pythonhosted.org/packages/b7/b9/c538f279a4e237a006a2c98387d081e9eb060d203d8ed34467cc0f0b9b53/packaging-26.0-py3-none-any.whl (from https://pypi.org/simple/packaging/) (requires-python:>=3.8), version: 26.0
    Found link https://files.pythonhosted.org/packages/65/ee/299d360cdc32edc7d2cf530f3accf79c4fca01e96ffc950d8a52213bd8e4/packaging-26.0.tar.gz (from https://pypi.org/simple/packaging/) (requires-python:>=3.8), version: 26.0
    Found link https://files.pythonhosted.org/packages/7a/c2/920ef838e2f0028c8262f16101ec09ebd5969864e5a64c4c05fad0617c56/packaging-26.1-py3-none-any.whl (from https://pypi.org/simple/packaging/) (requires-python:>=3.8), version: 26.1
    Found link https://files.pythonhosted.org/packages/df/de/0d2b39fb4af88a0258f3bac87dfcbb48e73fbdea4a2ed0e2213f9a4c2f9a/packaging-26.1.tar.gz (from https://pypi.org/simple/packaging/) (requires-python:>=3.8), version: 26.1
    Found link https://files.pythonhosted.org/packages/df/b2/87e62e8c3e2f4b32e5f

In [ ]:
# restarting session해서 requirements_filtered.txt가 사라지지 않게 하는 방법
%cd /content/MusiConGen
!grep -v -iE "^(xformers|torch|torchaudio|torchvision|torchtext)(==|>=|<)?" requirements.txt > /content/requirements_filtered.txt
!sed -i 's/^av==11.0.0/av/' /content/requirements_filtered.txt
!sed -i 's/^flashy==0.0.1/flashy==0.0.2/' /content/requirements_filtered.txt
!cat /content/requirements_filtered.txt

/content/MusiConGen
av
einops
flashy==0.0.2
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [ ]:
!pip install -r /content/requirements_filtered.txt -v 2>&1 | tail -100

    Created temporary directory: /tmp/pip-metadata-4t5qv_5l
  Created temporary directory: /tmp/pip-unpack-abdlqch3
  Looking up "https://files.pythonhosted.org/packages/18/ad/ec41343a49a0371ea40daf37b1ba2c11333cdd121cb378161635d14b9750/setuptools-59.2.0-py3-none-any.whl" in the cache
  No cache entry available
  No cache entry available
  https://files.pythonhosted.org:443 "GET /packages/18/ad/ec41343a49a0371ea40daf37b1ba2c11333cdd121cb378161635d14b9750/setuptools-59.2.0-py3-none-any.whl HTTP/1.1" 200 952017
  Ignoring unknown cache-control directive: immutable
  Updating cache with response from "https://files.pythonhosted.org/packages/18/ad/ec41343a49a0371ea40daf37b1ba2c11333cdd121cb378161635d14b9750/setuptools-59.2.0-py3-none-any.whl"
  etag object cached for 1209600 seconds
  Caching due to etag
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.0/952.0 kB 20.3 MB/s eta 0:00:00
  Looking up "https://files.pythonhosted.org/packages/04/80/cad93b40262f5d09f6de82adbee452fd43cdff60830b5

In [ ]:
# python 호환문제로 miniconda로 실행 환경 변경
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /content/miniconda3
!/content/miniconda3/bin/conda --version

PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3
conda 26.7.1


In [ ]:
# miniconda3의 격리된 환경 설정
!/content/miniconda3/bin/conda create -n stemremix python=3.11 -y

Jupyter detected...

CondaToSNonInteractiveError: Terms of Service have not been accepted for the following channels. Please accept or remove them before proceeding:
    - https://repo.anaconda.com/pkgs/main
    - https://repo.anaconda.com/pkgs/r

To accept these channels' Terms of Service, run the following commands:
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

For information on safely removing channels from your conda configuration,
please see the official documentation:

    https://www.anaconda.com/docs/tools/working-with-conda/channels



In [ ]:
 # 저장소 약관 동의
!/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [ ]:
# 다시 격리된 환경 설정 및 실행
!/content/miniconda3/bin/conda create -n stemremix python=3.11 -y

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / - \ | / - \ | / - \ | done
Channels:
 - defaults
Platform: linux-64
Solving environment: | done


==> WARNING: A newer version of conda exists. <==
    current version: 26.7.1
    latest version: 26.7.2

Please update conda by running

    $ conda self update



## Package Plan ##

  environment location: /content/miniconda3/envs/stemremix

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libexpat-2.8.4             |       h7354ed3_0         128 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    openssl-3.5.8              |       h1b28b03_0         5.5 MB
    packaging-26.3             |  py311h06a4308_0         380 KB
    pip-26.2.1                 |     pyhc872135_0         1.

In [ ]:
# miniconda 설치 -> gpu, pythohn 버전 확인
!/content/miniconda3/envs/stemremix/bin/python --version
!/content/miniconda3/envs/stemremix/bin/python -c "import sys; print(sys.executable)"
!nvidia-smi

Python 3.11.16
/content/miniconda3/envs/stemremix/bin/python
Mon Sep 14 07:20:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |           

In [ ]:
# stemremix 환경 안에 cuda 13.0에 맞는 torch 설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 23.1 MB/s  0:00:16
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 129.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 412.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 147.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 40.9 MB/s  0:00:11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 58.9 MB/s  0:00:05
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 170.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 175.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 181.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 175.1 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 158

In [ ]:
#gpu 인식여부 확인
!/content/miniconda3/envs/stemremix/bin/python -c "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'gpu 없음')"

/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:295: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
2.5.1+cu121
True
Tesla T4


In [ ]:
# MusiConGen 파이프라인에 numpy 필수 -> 설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 53.0 MB/s  0:00:00


In [ ]:
# numpy 설치 확인
!/content/miniconda3/envs/stemremix/bin/python -c "import numpy; print(numpy.__version__)"

2.4.6


### requirements_filtered.txt 재설치

In [ ]:
# 파일 확인
!cat /content/requirements_filtered.txt

av
einops
flashy==0.0.2
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [ ]:
# 설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install -r /content/requirements_filtered.txt

  Using cached av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (5.0 kB)
  Using cached flashy-0.0.2.tar.gz (72 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached hydra_core-1.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached hydra_colorlog-1.2.0-py3-none-any.whl.metadata (949 bytes)
  Using cached julius-0.2.8-py3-none-any.whl.metadata (7.6 kB)
  Using cached num2words-0.5.14-py3-none-any.whl.metadata (13 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 103.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ...

In [ ]:
# 설치된 버전 확인
!/content/miniconda3/envs/stemremix/bin/python -m pip list

Package                  Version
------------------------ -----------
annotated-types          0.8.0
antlr4-python3-runtime   4.8
audioread                3.1.0
av                       18.1.0
blis                     0.7.11
catalogue                2.0.10
certifi                  2026.7.22
cffi                     2.1.1
charset-normalizer       3.5.1
click                    8.5.0
cloudpickle              3.1.2
colorlog                 6.12.0
confection               0.1.5
cymem                    2.0.13
decorator                5.3.1
demucs                   4.1.0
docopt                   0.6.2
dora_search              0.1.13
einops                   0.8.2
encodec                  0.1.1
filelock                 3.32.3
flashy                   0.0.2
fsspec                   2026.7.0
hf-xet                   1.6.0
huggingface_hub          0.36.2
hydra-colorlog           1.2.0
hydra-core               1.1.0
idna                     3.19
Jinja2                   3.1.6
joblib             

In [ ]:
# 버전 파일로 저장
!/content/miniconda3/envs/stemremix/bin/python -m pip freeze > /content/stemremix_installed_versions.txt
!cat /content/stemremix_installed_versions.txt

annotated-types==0.8.0
antlr4-python3-runtime==4.8
audioread==3.1.0
av==18.1.0
blis==0.7.11
catalogue==2.0.10
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.1
click==8.5.0
cloudpickle==3.1.2
colorlog==6.12.0
confection==0.1.5
cymem==2.0.13
decorator==5.3.1
demucs==4.1.0
docopt==0.6.2
dora_search==0.1.13
einops==0.8.2
encodec==0.1.1
filelock==3.32.3
flashy==0.0.2
fsspec==2026.7.0
hf-xet==1.6.0
huggingface_hub==0.36.2
hydra-colorlog==1.2.0
hydra-core==1.1.0
idna==3.19
Jinja2==3.1.6
joblib==1.6.0
julius==0.2.8
lameenc==1.8.4
langcodes==3.5.1
lazy-loader==0.5
librosa==0.11.0
llvmlite==0.49.0
MarkupSafe==3.0.3
mpmath==1.3.0
msgpack==1.2.2
murmurhash==1.0.15
narwhals==2.26.0
networkx==3.6.1
num2words==0.5.14
numba==0.67.0
numpy==1.24.4
nvidia-cublas-cu12==12.1.3.1
nvidia-cuda-cupti-cu12==12.1.105
nvidia-cuda-nvrtc-cu12==12.1.105
nvidia-cuda-runtime-cu12==12.1.105
nvidia-cudnn-cu12==9.1.0.70
nvidia-cufft-cu12==11.0.2.54
nvidia-curand-cu12==10.3.2.106
nvidia-cusolver-cu12==11.4.5.107
n

In [ ]:
# drvie mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#MusiConGen 폴더 원본 확인
%cd /content/MusiConGen
!git status

/content/MusiConGen
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	requirements_filtered.txt
	requirements_no_xformers.txt

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!ls -la /content/drive/MyDrive/stem-remix-assistant/
!ls -la /content/requirements_filtered.txt /content/stemremix_installed_versions.txt

total 13980
drwx------ 3 root root     4096 Aug 14 04:06  .
drwx------ 5 root root     4096 Sep 14 07:24  ..
-rw------- 1 root root 14300011 Aug 28 08:52 'Copy of 01_setup_and_test.ipynb'
drwx------ 2 root root     4096 Aug 14 04:06  outputs
-rw------- 1 root root      216 Sep  2 08:11  requirements_filtered.txt
-rw------- 1 root root     1878 Sep  2 08:11  stemremix_installed_versions.txt
-rw-r--r-- 1 root root  216 Sep 14 07:18 /content/requirements_filtered.txt
-rw-r--r-- 1 root root 1879 Sep 14 07:24 /content/stemremix_installed_versions.txt


In [ ]:
# /content/의 하위 .txt를 복사
!cp -v /content/requirements_filtered.txt /content/drive/MyDrive/stem-remix-assistant/
!cp -v /content/stemremix_installed_versions.txt /content/drive/MyDrive/stem-remix-assistant/
!ls -la /content/drive/MyDrive/stem-remix-assistant/

'/content/requirements_filtered.txt' -> '/content/drive/MyDrive/stem-remix-assistant/requirements_filtered.txt'
'/content/stemremix_installed_versions.txt' -> '/content/drive/MyDrive/stem-remix-assistant/stemremix_installed_versions.txt'
total 13980
drwx------  3 root root     4096 Sep 14 07:25  .
drwx------ 11 root root     4096 Sep 14 07:24  ..
-rw-------  1 root root 14300011 Aug 28 08:52 'Copy of 01_setup_and_test.ipynb'
drwx------  2 root root     4096 Aug 14 04:06  outputs
-rw-------  1 root root      216 Sep 14 07:25  requirements_filtered.txt
-rw-------  1 root root     1879 Sep 14 07:25  stemremix_installed_versions.txt


### audiocraft import

In [ ]:
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/audio.py", line 25, in <module>\n    from .audio_utils import f32_pcm, i16_pcm, normalize_audio\n  File "/content/MusiConGen/audiocraft/audiocraft/data/audio_utils.py", line 16, in <module>\n    from .chords import Chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/chords.py", line 33, in <module>\n    import pandas as pd\nModuleNotFoundError: No module named \'pandas\'\n'


In [ ]:
!ls /content/MusiConGen/
print("===============")
!ls /content/MusiConGen/audiocraft/
print("===============")
!ls /content/MusiConGen/audiocraft/audiocraft/

5_genre_songs_list.json  README.md
audiocraft		 requirements_filtered.txt
LICENSE			 requirements_no_xformers.txt
preproc			 requirements.txt
audiocraft  config  dataset  egs  export_weight.py  generate_chord_beat.py
adversarial	grids	     metrics  optim	   quantization  utils
data		__init__.py  models   __pycache__  solvers
environment.py	losses	     modules  py.typed	   train.py


In [ ]:
# pandas 의존성 확인 -> requirements_filterd.txt 기존에는 없음
!head -50 /content/MusiConGen/audiocraft/audiocraft/data/chords.py

# encoding: utf-8
"""
This module contains chord evaluation functionality.

It provides the evaluation measures used for the MIREX ACE task, and
tries to follow [1]_ and [2]_ as closely as possible.

Notes
-----
This implementation tries to follow the references and their implementation
(e.g., https://github.com/jpauwels/MusOOEvaluator for [2]_). However, there
are some known (and possibly some unknown) differences. If you find one not
listed in the following, please file an issue:

 - Detected chord segments are adjusted to fit the length of the annotations.
   In particular, this means that, if necessary, filler segments of 'no chord'
   are added at beginnings and ends. This can result in different segmentation
   scores compared to the original implementation.

References
----------
.. [1] Christopher Harte, "Towards Automatic Extraction of Harmony Information
       from Music Signals." Dissertation,
       Department for Electronic Engineering, Queen Mary University of London,
  

In [ ]:
# 코드가 특정 구버전 api에 의존하는지 확인
!grep -n "pd\." /content/MusiConGen/audiocraft/audiocraft/data/chords.py

464:        df = pd.DataFrame(data=entry[['root', 'is_major']])


In [ ]:
# pandas latest ver. installation
!/content/miniconda3/envs/stemremix/bin/python -m pip install pandas

  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 113.0 MB/s  0:00:00
Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]


In [ ]:
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/audio_dataset.py", line 33, in <module>\n    import dora\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/dora/__init__.py", line 68, in <module>\n    import hydra\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/hydra/__init__.py", line 5, in <module>\n    from hydra import utils\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/hydra/utils.py", line 8, in <module>\n    import hydra._internal.instantiate._instantiate2\n  File "/conte

In [ ]:
# 재검증 -> hydra-core 호환성 문제
!/content/miniconda3/envs/stemremix/bin/python -m pip install "hydra-core==1.3.2" --force-reinstall

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
Using cached pyyaml-6.0.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (806 kB)
Using cached packaging-26.3-py3-none-any.whl (129 kB)
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=a13d9478bee8b91edac0966752dc1bc7012a363138e093211ca69121761c1a4b
  Stored in directory: /root/.cache/pip/wheels/1a/97/32/461f837398029ad76911109f07047fde1d7b661a147c7c56d1
Successfully built antlr4-python3-runtime
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.8
    Uninstalling antlr4-python3-runtime-4.8:
      Successfully 

In [ ]:
# hydra-core 다운그레이드 이후 재검증
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/info_audio_dataset.py", line 19, in <module>\n    from ..modules.conditioners import SegmentWithAttributes, ConditioningAttributes\n  File "/content/MusiConGen/audiocraft/audiocraft/modules/__init__.py", line 22, in <module>\n    from .transformer import StreamingTransformer\n  File "/content/MusiConGen/audiocraft/audiocraft/modules/transformer.py", line 23, in <module>\n    from xformers import ops\nModuleNotFoundError: No module named \'xformers\'\n'


In [ ]:
# transformers가 사용하는 xformers 확인 -> 필수/선택 의존성 여부 확인
!sed -n '1,40p' /content/MusiConGen/audiocraft/audiocraft/modules/transformer.py

# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.

"""
Transformer model, with streaming support, xformer attention support
and easy causal attention with a potentially finite receptive field.

See `StreamingTransformer` for more information.

Unlike regular PyTorch Transformer, we make the hard choice that batches are first.
"""

import typing as tp

from einops import rearrange
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from xformers import ops

from .rope import RotaryEmbedding
from .streaming import StreamingModule

_efficient_attention_backend: str = 'torch'


def set_efficient_attention_backend(backend: str = 'torch'):
    # Using torch by default, it seems a bit faster on older P100 GPUs (~20% faster).
    global _efficient_attention_backen

In [ ]:
# ops 모듈 호출 위치 확인
!grep -n "ops\." /content/MusiConGen/audiocraft/audiocraft/modules/transformer.py

373:                    q, k, v = ops.unbind(packed, dim=2)
407:                    x = ops.memory_efficient_attention(q, k, v, attn_mask, p=p)


In [ ]:
!sed -n '355,410p' /content/MusiConGen/audiocraft/audiocraft/modules/transformer.py

                k = nn.functional.linear(key, self.in_proj_weight[dim: 2 * dim], bias_k)
                v = nn.functional.linear(value, self.in_proj_weight[2 * dim:], bias_v)
                if self.qk_layer_norm is True:
                    q = self.q_layer_norm(q)
                    k = self.k_layer_norm(k)
                q, k, v = [rearrange(x, f"b t (h d) -> {layout}", h=self.num_heads) for x in [q, k, v]]
            else:
                if not _is_profiled():
                    # profiling breaks that propertysomehow.
                    assert query is key, "specialized implementation"
                    assert value is key, "specialized implementation"
                projected = nn.functional.linear(query, self.in_proj_weight, self.in_proj_bias)
                if self.kv_repeat == 1:
                    if time_dim == 2:
                        bound_layout = "b h p t d"
                    else:
                        bound_layout = "b t p h d"
                    pac

In [ ]:
# xformers installation
!/content/miniconda3/envs/stemremix/bin/python -m pip install xformers --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 31.7 MB/s  0:00:00


In [ ]:
# audiocraft 재검증
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/info_audio_dataset.py", line 19, in <module>\n    from ..modules.conditioners import SegmentWithAttributes, ConditioningAttributes\n  File "/content/MusiConGen/audiocraft/audiocraft/modules/conditioners.py", line 6, in <module>\n    import pretty_midi\nModuleNotFoundError: No module named \'pretty_midi\'\n'


In [ ]:
# import 구문 전체 grep해서 버전 확인
!grep -rhE "^(import |from )" /content/MusiConGen/audiocraft/audiocraft/ | sort -u

grep: /content/MusiConGen/audiocraft/audiocraft/models/__pycache__/builders.cpython-311.pyc: binary file matches
grep: /content/MusiConGen/audiocraft/audiocraft/solvers/__pycache__/builders.cpython-311.pyc: binary file matches
from abc import ABC, abstractmethod
from audiocraft.data.audio import audio_write
from audiocraft.data.audio_utils import convert_audio
from audiocraft import __version__
from audiocraft.modules.transformer import StreamingTransformer, create_sin_embedding
from .audio_dataset import AudioDataset, AudioMeta
from .audiogen import AudioGen
from .audiogen import AudioGenSolver
from .audio import audio_read, audio_info
from .audio_utils import convert_audio
from .audio_utils import f32_pcm, i16_pcm, normalize_audio
from .balancer import Balancer
from .._base_explorers import BaseExplorer
from .base import BaseQuantizer, DummyQuantizer, QuantizedResult
from .base import BaseQuantizer, QuantizedResult
from .base import MultiDiscriminator, MultiDiscriminatorOutputType
fr

In [ ]:
# 현재 stemremix 환경에 설치된 패키지 목록 확인
!/content/miniconda3/envs/stemremix/bin/python -m pip list --format=freeze

annotated-types==0.8.0
antlr4-python3-runtime==4.9.3
audioread==3.1.0
av==18.1.0
blis==0.7.11
catalogue==2.0.10
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.1
click==8.5.0
cloudpickle==3.1.2
colorlog==6.12.0
confection==0.1.5
cymem==2.0.13
decorator==5.3.1
demucs==4.1.0
docopt==0.6.2
dora_search==0.1.13
einops==0.8.2
encodec==0.1.1
filelock==3.32.3
flashy==0.0.2
fsspec==2026.7.0
hf-xet==1.6.0
huggingface_hub==0.36.2
hydra-colorlog==1.2.0
hydra-core==1.3.2
idna==3.19
Jinja2==3.1.6
joblib==1.6.0
julius==0.2.8
lameenc==1.8.4
langcodes==3.5.1
lazy-loader==0.5
librosa==0.11.0
llvmlite==0.49.0
MarkupSafe==3.0.3
mpmath==1.3.0
msgpack==1.2.2
murmurhash==1.0.15
narwhals==2.26.0
networkx==3.6.1
num2words==0.5.14
numba==0.67.0
numpy==2.4.6
nvidia-cublas-cu12==12.1.3.1
nvidia-cuda-cupti-cu12==12.1.105
nvidia-cuda-nvrtc-cu12==12.1.105
nvidia-cuda-runtime-cu12==12.1.105
nvidia-cudnn-cu12==9.1.0.70
nvidia-cufft-cu12==11.0.2.54
nvidia-curand-cu12==10.3.2.106
nvidia-cusolver-cu12==11.4.5.107


In [ ]:
# 누락된 prety_middi, torchmetricx 설치후 재검증
!/content/miniconda3/envs/stemremix/bin/python -m pip install pretty_midi torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 18.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 16.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [torchmetrics]


In [ ]:
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/info_audio_dataset.py", line 19, in <module>\n    from ..modules.conditioners import SegmentWithAttributes, ConditioningAttributes\n  File "/content/MusiConGen/audiocraft/audiocraft/modules/conditioners.py", line 21, in <module>\n    import spacy\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/spacy/__init__.py", line 6, in <module>\n    from .errors import setup_default_warnings\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/spacy/errors.py", line 3, in

In [ ]:
# numpy 버전이 호환이 안됨 --> 언제 버전 바뀌었는지 확인
!/content/miniconda3/envs/stemremix/bin/python -m pip show numpy
print("==============================================")
!/content/miniconda3/envs/stemremix/bin/python -m pip show thinc

Name: numpy
Version: 2.4.6
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License-Expression: BSD-3-Clause AND 0BSD AND MIT AND Zlib AND CC0-1.0
Location: /content/miniconda3/envs/stemremix/lib/python3.11/site-packages
Requires: 
Required-by: blis, encodec, flashy, librosa, numba, pandas, pretty_midi, pystoi, scikit-learn, scipy, soundfile, soxr, spacy, thinc, torchmetrics, transformers, xformers
Name: thinc
Version: 8.1.12
Summary: A refreshing functional take on deep learning, compatible with your favorite libraries
Home-page: https://github.com/explosion/thinc
Author: Explosion
Author-email: contact@explosion.ai
License: MIT
Location: /content/miniconda3/envs/stemremix/lib/python3.11/site-packages
Requires: blis, catalogue, confection, cymem, murmurhash, numpy, packaging, preshed, pydantic, setuptools, srsly, wasabi
Required-by: spacy


In [ ]:
# 호환버전 numpy로 재설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install "numpy==1.24.4" --force-reinstall

  Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 3.0.5 requires numpy>=1.26.0; python_version < "3.14", but you have numpy 1.24.4 which is incompatible.


In [ ]:
# thic과 pandas 모두 호환되는 버전으로 재설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install "numpy==1.26.4" --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 39.3 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4


In [ ]:
# 재검증 완료 -> thic, pandas 모두와 호환됨
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 0
STDOUT >>> 'OK: /content/MusiConGen/audiocraft/audiocraft/__init__.py\n'
STDERR >>> ''


In [ ]:
# stemremix환경에 ipykernel 설치/등록
# stemremix conda 환경을 Colab의 Jupyter 커널로 등록
# 실행 후: 상단 메뉴 "런타임 > 런타임 유형 변경"이 아니라,
#          우측 상단 "연결됨" 옆 화살표 또는 "런타임 > 다른 런타임에 연결" 근처의
#          커널 선택 메뉴(또는 노트북 설정의 "Kernel" 드롭다운)에서
#          "Python (stemremix)" 을 선택해야 적용됨.
# Colab 버전에 따라 커널 드롭다운 위치가 다를 수 있음 — 안 보이면
# 상단 메뉴에서 "런타임 > 런타임 유형 변경" 옆에 있는 커널 아이콘을 확인.

!/content/miniconda3/envs/stemremix/bin/python -m pip install -q ipykernel
!/content/miniconda3/envs/stemremix/bin/python -m ipykernel install --user --name stemremix --display-name "Python (stemremix)"

print("등록 완료. 이제 Colab에서 커널을 'Python (stemremix)'로 전환한 뒤,")
print("아래 확인 셀을 새 커널에서 실행해서 stemremix 환경이 맞는지 검증할 것.")

Installed kernelspec stemremix in /root/.local/share/jupyter/kernels/stemremix
등록 완료. 이제 Colab에서 커널을 'Python (stemremix)'로 전환한 뒤,
아래 확인 셀을 새 커널에서 실행해서 stemremix 환경이 맞는지 검증할 것.


In [ ]:
# 새 커널에서 실행 -> audiocraft import 성공 여부 확인
# 커널을 "Python (stemremix)"로 전환한 뒤 이 셀을 실행해서 확인
# sys.executable이 /content/miniconda3/envs/stemremix/bin/python 이어야 정상

import sys
print("현재 커널 파이썬:", sys.executable)

import numpy, torch, xformers
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("xformers:", xformers.__version__)
print("CUDA available:", torch.cuda.is_available())

# audiocraft 경로를 이번엔 커널 자체에 등록 (매 셀마다 sys.path.insert 반복 안 하도록)
sys.path.insert(0, "/content/MusiConGen/audiocraft")
import audiocraft
print("audiocraft OK:", audiocraft.__file__)

현재 커널 파이썬: /usr/bin/python3


ModuleNotFoundError: No module named 'xformers'

In [4]:
# colab에서 kernerpspec과 무관하게 연결 안됨 -> 스펙 되돌린 후 재확인
!ls /content/miniconda3/envs/stemremix 2>&1 | head -5
print("================================")
!/content/miniconda3/envs/stemremix/bin/python -c "import numpy, xformers, pandas; print(numpy.__version__, xformers.__version__, pandas.__version__)"

bin
compiler_compat
conda-meta
etc
include
1.26.4 0.0.29.post1 3.0.5


In [6]:
# audiocraft 재검증
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 0
STDOUT >>> 'OK: /content/MusiConGen/audiocraft/audiocraft/__init__.py\n'
STDERR >>> ''


### 저장소 연결 끊겼을 때

In [1]:
import os
print(os.path.exists("/content/miniconda3/envs/stemremix"))

False


In [2]:
import os
if not os.path.exists("/content/MusiConGen"):
    !git clone https://github.com/YatingMusic/MusiConGen.git /content/MusiConGen

Cloning into '/content/MusiConGen'...
remote: Enumerating objects: 449, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 449 (delta 8), reused 6 (delta 6), pack-reused 417 (from 1)
Receiving objects: 100% (449/449), 112.34 MiB | 33.75 MiB/s, done.
Resolving deltas: 100% (70/70), done.


In [3]:
import subprocess

result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import numpy, torch, xformers, pandas, hydra, flashy; '
     'print("numpy:", numpy.__version__); '
     'print("torch:", torch.__version__); '
     'print("xformers:", xformers.__version__); '
     'print("pandas:", pandas.__version__); '
     'print("hydra:", hydra.__version__); '
     'print("flashy:", flashy.__version__); '
     'print("CUDA available:", torch.cuda.is_available())'],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("returncode:", result.returncode)

FileNotFoundError: [Errno 2] No such file or directory: '/content/miniconda3/envs/stemremix/bin/python'

In [4]:
import os
print("miniconda3 폴더 존재:", os.path.exists("/content/miniconda3"))
print("conda 실행파일 존재:", os.path.exists("/content/miniconda3/bin/conda"))

miniconda3 폴더 존재: False
conda 실행파일 존재: False


In [5]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
!bash /content/miniconda.sh -b -p /content/miniconda3

PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3


In [6]:
import os
print("설치 확인:", os.path.exists("/content/miniconda3/bin/conda"))

설치 확인: True


In [7]:
# stemremix 환경 생성(python 3.11)
!/content/miniconda3/bin/conda create -n stemremix python=3.11 -y

Jupyter detected...

CondaToSNonInteractiveError: Terms of Service have not been accepted for the following channels. Please accept or remove them before proceeding:
    - https://repo.anaconda.com/pkgs/main
    - https://repo.anaconda.com/pkgs/r

To accept these channels' Terms of Service, run the following commands:
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

For information on safely removing channels from your conda configuration,
please see the official documentation:

    https://www.anaconda.com/docs/tools/working-with-conda/channels



In [8]:
# 완료 확인
import os
print("stemremix 환경 python 존재:", os.path.exists("/content/miniconda3/envs/stemremix/bin/python"))

stemremix 환경 python 존재: False


In [9]:
# Anaconda 정책 이슈 -> 채널 동의
!/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [10]:
# 환경 생성 재시도
!/content/miniconda3/bin/conda create -n stemremix python=3.11 -y

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / - \ | done
Channels:
 - defaults
Platform: linux-64
Solving environment: \ done


==> WARNING: A newer version of conda exists. <==
    current version: 26.7.1
    latest version: 26.7.2

Please update conda by running

    $ conda self update



## Package Plan ##

  environment location: /content/miniconda3/envs/stemremix

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libexpat-2.8.4             |       h7354ed3_0         128 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    openssl-3.5.8              |       h1b28b03_0         5.5 MB
    packaging-26.3             |  py311h06a4308_0         380 KB
    pip-26.2.1                 |     pyhc872135_0         1.1 MB
    python-3.11.16         

In [11]:
# 재확인
import os
print("stemremix 환경 python 존재:", os.path.exists("/content/miniconda3/envs/stemremix/bin/python"))

stemremix 환경 python 존재: True


In [12]:
# 검증된 조합으로 패키지 재설치
STEMREMIX_PIP = "/content/miniconda3/envs/stemremix/bin/pip"

!{STEMREMIX_PIP} install torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!{STEMREMIX_PIP} install numpy==1.26.4
!{STEMREMIX_PIP} install hydra-core==1.3.2 hydra_colorlog
!{STEMREMIX_PIP} install flashy==0.0.2
!{STEMREMIX_PIP} install xformers==0.0.29.post1 --index-url https://download.pytorch.org/whl/cu121
!{STEMREMIX_PIP} install pandas==3.0.5
!{STEMREMIX_PIP} install av julius einops num2words sentencepiece spacy==3.6.1 \
    tqdm demucs librosa soundfile torchmetrics encodec protobuf pesq pystoi \
    transformers==4.31.0 pretty_midi huggingface_hub

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 35.7 MB/s  0:00:12
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 13.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 266.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 391.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 413.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 30.4 MB/s  0:00:13
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 55.7 MB/s  0:00:05
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 282.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 395.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 233.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 84.0 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 215.

In [13]:
# 검증 커맨드 확인
import subprocess

result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import numpy, torch, xformers, pandas, hydra, flashy; '
     'print("numpy:", numpy.__version__); '
     'print("torch:", torch.__version__); '
     'print("xformers:", xformers.__version__); '
     'print("pandas:", pandas.__version__); '
     'print("hydra:", hydra.__version__); '
     'print("flashy:", flashy.__version__); '
     'print("CUDA available:", torch.cuda.is_available())'],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("returncode:", result.returncode)

STDOUT: numpy: 1.26.4
torch: 2.5.1+cu121
xformers: 0.0.29.post1
pandas: 3.0.5
hydra: 1.3.2
flashy: 0.0.2
CUDA available: True

STDERR: 
returncode: 0


In [14]:
# bottleneck 재검증 -> audiocraft import 검증 (stemremix 환경에서)
import subprocess

result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); '
     'import audiocraft; print("audiocraft OK:", audiocraft.__file__); '
     'from audiocraft.data.audio import audio_write; print("audio_write OK")'],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("returncode:", result.returncode)

STDOUT: audiocraft OK: /content/MusiConGen/audiocraft/audiocraft/__init__.py
audio_write OK

STDERR: 
returncode: 0


## 2. 체크포인트 다운로드

공식 안내대로 HuggingFace(`Cyan0731/MusiConGen`)에서 `compression_state_dict.bin`, `state_dict.bin`을 받아
`audiocraft/ckpt/musicongen/`에 배치. (README 기준 확인된 절차 — 임의 추정 아님)

In [15]:
# =========================================================
# audiocraft 실제 clone 경로 확인용 스크립트
# =========================================================

import os

# 1. /content 아래에서 audiocraft 관련 폴더/파일 전부 탐색
print("=== /content 하위에서 'audiocraft' 관련 항목 검색 ===")
for root, dirs, files in os.walk("/content"):
    # 너무 깊은 하위 폴더(예: site-packages 등)는 건너뛰어 속도 확보
    if root.count(os.sep) - "/content".count(os.sep) > 4:
        dirs[:] = []
        continue
    for d in dirs:
        if "audiocraft" in d.lower():
            print("DIR :", os.path.join(root, d))
    for f in files:
        if f == "__init__.py" and "audiocraft" in root.lower():
            print("FILE:", os.path.join(root, f))

# 2. 후보 경로들 직접 체크 - __init__.py 존재 여부로 진짜 패키지 루트 판별
candidates = [
    "/content/MusiConGen/audiocraft",
    "/content/MusiConGen/audiocraft/audiocraft",
    "/content/audiocraft",
    "/content/audiocraft/audiocraft",
]

print("\n=== 후보 경로 검증 (진짜 패키지 루트는 여기에 audiocraft/__init__.py가 있어야 함) ===")
for c in candidates:
    init_path = os.path.join(c, "audiocraft", "__init__.py")
    exists = os.path.exists(init_path)
    print(f"{c}  ->  {init_path} 존재? {exists}")

print("\n주의: AUDIOCRAFT_ROOT에 넣을 값은 'audiocraft' 패키지 폴더의 '부모' 경로입니다.")
print("즉 위에서 존재(True)로 나온 __init__.py 경로에서, 'audiocraft/__init__.py' 앞부분까지가 AUDIOCRAFT_ROOT입니다.")

=== /content 하위에서 'audiocraft' 관련 항목 검색 ===
DIR : /content/MusiConGen/audiocraft
DIR : /content/MusiConGen/audiocraft/audiocraft
FILE: /content/MusiConGen/audiocraft/audiocraft/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/optim/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/solvers/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/models/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/utils/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/metrics/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/losses/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/grids/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/quantization/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/data/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/modules/__init__.py
FILE: /content/MusiConGen/audiocraft/audiocraft/adversarial/__init__.py

=== 후보 경로 검증 (진짜 패키지 루트는 여기에 audiocraft/__init__.py가 있어야 함) ===


In [16]:
# =========================================================
# MusiConGen 체크포인트 재다운로드 + 모델 로드
# (audiocraft 패키지 이름과 겹치지 않는 디렉토리 사용)
# =========================================================

import os, sys

# 1. 작업 디렉토리 확인 (반드시 저장소 루트인지 체크)
print("현재 작업 디렉토리:", os.getcwd())

# 2. 체크포인트 저장 폴더 - "audiocraft"라는 이름과 절대 겹치지 않게 설정
#    (이전 사고: audiocraft/ckpt/musicongen -> 네임스페이스 패키지 충돌 발생)
ckpt_dir = os.path.abspath("./musicongen_checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)
print("체크포인트 저장 경로:", ckpt_dir)

# 3. huggingface_hub으로 체크포인트 다운로드
from huggingface_hub import hf_hub_download

# 실제 레포 이름/파일명은 이전에 쓰던 것으로 맞추기
REPO_ID = "Cyan0731/MusiConGen"
FILES = ["compression_state_dict.bin", "state_dict.bin"]

for fname in FILES:
    local_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=fname,
        local_dir=ckpt_dir,
    )
    print(f"다운로드 완료: {local_path}")

# 4. import 전에 sys.modules 캐시 정리 (혹시 이전에 잘못된 audiocraft가 캐싱됐을 경우 대비)
for mod in list(sys.modules):
    if mod == "audiocraft" or mod.startswith("audiocraft."):
        del sys.modules[mod]

# 5. 진짜 audiocraft 패키지 경로를 최우선으로 등록
#    (경로는 실제 저장소 clone 위치에 맞게 수정)
AUDIOCRAFT_ROOT = "/content/MusiConGen/audiocraft"  # 실제 경로로 교체
if AUDIOCRAFT_ROOT not in sys.path:
    sys.path.insert(0, AUDIOCRAFT_ROOT)

import audiocraft
print("audiocraft 위치 확인:", audiocraft.__file__)  # None이면 네임스페이스 충돌 재발 - 즉시 중단

assert audiocraft.__file__ is not None, "네임스페이스 패키지 충돌 재발! 폴더명 다시 확인 필요"

# 6. 모델 로드
from audiocraft.models import MusicGen

model = MusicGen.get_pretrained(ckpt_dir)  # 로컬 체크포인트 경로 사용
print("모델 로드 완료")

현재 작업 디렉토리: /content
체크포인트 저장 경로: /content/musicongen_checkpoints


compression_state_dict.bin: reconstructing file:   0%|          |  0.00B /   589B            

compression_state_dict.bin: downloading bytes:           |  0.00B            

다운로드 완료: /content/musicongen_checkpoints/compression_state_dict.bin


state_dict.bin: reconstructing file:   0%|          |  0.00B / 2.77GB            

state_dict.bin: downloading bytes:           |  0.00B            

다운로드 완료: /content/musicongen_checkpoints/state_dict.bin


ModuleNotFoundError: No module named 'av'

In [17]:
# =========================================================
# audiocraft import에 필요한 의존성 일괄 설치
# (이전에 검증된 버전 조합 - whack-a-mole 방지)
# =========================================================

# av, julius 등 audiocraft.data 서브모듈이 요구하는 패키지들을 한 번에 설치
!pip install -q av julius einops num2words sentencepiece spacy==3.6.1 \
    tqdm demucs librosa soundfile torchmetrics encodec protobuf pesq pystoi transformers==4.31.0

# numpy / hydra-core / flashy는 이전에 ABI 충돌 겪었던 조합이므로 검증된 버전으로 고정
!pip install -q --upgrade numpy==1.26.4
!pip install -q --upgrade hydra-core==1.3.2 hydra_colorlog
!pip install -q --upgrade flashy==0.0.2

print("설치 완료 - 이제 아래 셀에서 import audiocraft 다시 시도")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 100.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

In [19]:
import sys # ==> 여전히 에러 발생. 왜지?

# 4. import 전에 sys.modules 캐시 정리
for mod in list(sys.modules):
    if mod == "audiocraft" or mod.startswith("audiocraft."):
        del sys.modules[mod]

# 5. 진짜 audiocraft 패키지 경로를 최우선으로 등록
AUDIOCRAFT_ROOT = "/content/MusiConGen/audiocraft"
if AUDIOCRAFT_ROOT not in sys.path:
    sys.path.insert(0, AUDIOCRAFT_ROOT)

import audiocraft
print("audiocraft 위치 확인:", audiocraft.__file__)

assert audiocraft.__file__ is not None, "네임스페이스 패키지 충돌 재발! 폴더명 다시 확인 필요"

# 6. 모델 로드
from audiocraft.models import MusicGen

ckpt_dir = "/content/musicongen_checkpoints"  # 이미 다운로드된 경로 (os.getcwd() 결과가 /content였으므로)
model = MusicGen.get_pretrained(ckpt_dir)
print("모델 로드 완료")

ModuleNotFoundError: No module named 'av'

In [20]:
# =========================================================
# av 패키지 설치 상태 직접 확인 + 단독 재설치
# =========================================================

# 1. 현재 av가 정말 없는지, 어떤 파이썬에 설치하려 했는지 확인
import sys
print("현재 커널의 파이썬 실행 경로:", sys.executable)

try:
    import av
    print("av 이미 설치되어 있음:", av.__file__)
except ImportError as e:
    print("av import 실패:", e)

# 2. av만 단독으로 설치 - verbose 출력으로 실제 에러 확인
!pip install av --no-cache-dir -v 2>&1 | tail -n 60

현재 커널의 파이썬 실행 경로: /usr/bin/python3
av import 실패: No module named 'av'
Using pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
  Obtaining dependency information for av from https://files.pythonhosted.org/packages/27/3a/204dbfc3e08eb4cdc6e6ff57be02150bc44523ebdb50182d10025792ebd9/av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 336.2 MB/s eta 0:00:00
  changing mode of /usr/local/bin/pyav to 755


In [21]:
# audiocraft 재시도
import sys

for mod in list(sys.modules):
    if mod == "audiocraft" or mod.startswith("audiocraft."):
        del sys.modules[mod]

AUDIOCRAFT_ROOT = "/content/MusiConGen/audiocraft"
if AUDIOCRAFT_ROOT not in sys.path:
    sys.path.insert(0, AUDIOCRAFT_ROOT)

import audiocraft
print("audiocraft 위치 확인:", audiocraft.__file__)

ModuleNotFoundError: No module named 'julius'

In [22]:
# whack-a-moel 패턴 재등장
# subprocess.run으로 설치하면서 return code 직접체크한 것으로 변경

# =========================================================
# audiocraft 의존성 전체를 하나씩 설치 + 성공 여부 확인
# (한꺼번에 설치했을 때 실패가 묻히는 문제 방지)
# =========================================================

import subprocess, sys

packages = [
    "julius", "einops", "num2words", "sentencepiece", "spacy==3.6.1",
    "tqdm", "demucs", "librosa", "soundfile", "torchmetrics",
    "encodec", "protobuf", "pesq", "pystoi", "transformers==4.31.0",
]

failed = []
for pkg in packages:
    print(f"\n===== 설치 시도: {pkg} =====")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"!!! 실패: {pkg}")
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        failed.append(pkg)
    else:
        print(f"성공: {pkg}")

print("\n\n========== 최종 결과 ==========")
if failed:
    print("설치 실패한 패키지:", failed)
else:
    print("전부 설치 성공")


===== 설치 시도: julius =====
성공: julius

===== 설치 시도: einops =====
성공: einops

===== 설치 시도: num2words =====
성공: num2words

===== 설치 시도: sentencepiece =====
성공: sentencepiece

===== 설치 시도: spacy==3.6.1 =====
!!! 실패: spacy==3.6.1

  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


===== 설치 시도: tqdm =====
성공: tqdm

===== 설치 시도: demucs =====
성공: demucs

===== 설치 시도: librosa =====
성공: librosa

===== 설치 시도: soundfile =====
성공: soundfile

===== 설치 시도: torchmetrics =====
성공: torchmetrics

===== 설치 시도: encodec =====
성공: encodec

===== 설치 시도: protobuf =====
성공: proto

In [23]:
# 컴파일 로그 확인
# =========================================================
# spacy / transformers(tokenizers) 빌드 실패 원인 진단
# - 전체 로그를 파일로 저장해서 잘리지 않게 확인
# =========================================================

import subprocess, sys

def try_install(pkg, logfile):
    print(f"\n===== 설치 시도: {pkg} (전체 로그 -> {logfile}) =====")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-v", pkg],
        capture_output=True, text=True
    )
    with open(logfile, "w") as f:
        f.write("=== STDOUT ===\n")
        f.write(result.stdout)
        f.write("\n=== STDERR ===\n")
        f.write(result.stderr)
    print("returncode:", result.returncode)
    print(f"전체 로그 저장 완료: {logfile} (길이: {len(result.stdout) + len(result.stderr)}자)")
    return result.returncode

rc1 = try_install("spacy==3.6.1", "/content/spacy_install_log.txt")
rc2 = try_install("transformers==4.31.0", "/content/transformers_install_log.txt")

# 실제 컴파일러가 뱉은 에러 라인만 필터링해서 미리보기
import re

def show_real_errors(logfile):
    print(f"\n--- {logfile} 에서 핵심 에러 라인만 추출 ---")
    with open(logfile) as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if re.search(r"error[: ]|Error:|fatal error|ERROR:", line, re.IGNORECASE):
            print(line.rstrip())

show_real_errors("/content/spacy_install_log.txt")
show_real_errors("/content/transformers_install_log.txt")

print("\n\n필요하면 아래로 전체 로그 파일 내용을 직접 열어서 위아래 맥락을 더 볼 수 있음:")
print("!cat /content/spacy_install_log.txt")
print("!cat /content/transformers_install_log.txt")


===== 설치 시도: spacy==3.6.1 (전체 로그 -> /content/spacy_install_log.txt) =====
returncode: 1
전체 로그 저장 완료: /content/spacy_install_log.txt (길이: 10373115자)

===== 설치 시도: transformers==4.31.0 (전체 로그 -> /content/transformers_install_log.txt) =====
returncode: 1
전체 로그 저장 완료: /content/transformers_install_log.txt (길이: 467319자)

--- /content/spacy_install_log.txt 에서 핵심 에러 라인만 추출 ---
      ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
      Error compiling Cython file:
      # intentionally created compiler error that only triggers on Cython < 3.0.0
      /tmp/pip-build-env-fdb3nwvm/overlay/local/lib/python3.13/dist-packages/numpy/__init__.pxd:12:13: Error in compile-time expression: ValueError: invalid literal for int() with base 10: 'Build aborted: the NumPy Cython headers require Cython 3.0.0 or newer.'
      Error compiling Cython file:
      Error compiling Cython fi

In [24]:
# =========================================================
# 저장소 안에서 spacy가 실제로 어디에, 왜 필요한지 확인
# =========================================================

import subprocess

# 1. MusiConGen / audiocraft 저장소 안에서 spacy 사용처 검색
print("=== 코드에서 'spacy' 직접 import/사용 검색 ===")
result = subprocess.run(
    ["grep", "-rn", "spacy", "/content/MusiConGen", "--include=*.py"],
    capture_output=True, text=True
)
print(result.stdout if result.stdout else "(코드 내 직접 사용 없음)")

# 2. requirements.txt 등에서 spacy가 어디서 요구되는지 확인
print("\n=== requirements 파일들에서 'spacy' 언급 검색 ===")
result2 = subprocess.run(
    ["grep", "-rn", "-i", "spacy", "/content/MusiConGen",
     "--include=*.txt", "--include=*.cfg", "--include=*.toml"],
    capture_output=True, text=True
)
print(result2.stdout if result2.stdout else "(requirements 파일에 명시적 언급 없음)")

# 3. spacy가 num2words나 다른 패키지의 하위 의존성으로 딸려오는 것인지 확인
print("\n=== 어떤 패키지가 spacy를 요구하는지 (설치된 패키지 기준) ===")
result3 = subprocess.run(
    ["pip", "show", "spacy"],
    capture_output=True, text=True
)
print(result3.stdout, result3.stderr)

=== 코드에서 'spacy' 직접 import/사용 검색 ===
/content/MusiConGen/audiocraft/audiocraft/modules/conditioners.py:21:import spacy
/content/MusiConGen/audiocraft/audiocraft/modules/conditioners.py:269:            self.nlp = spacy.load(language)
/content/MusiConGen/audiocraft/audiocraft/modules/conditioners.py:271:            spacy.cli.download(language)  # type: ignore
/content/MusiConGen/audiocraft/audiocraft/modules/conditioners.py:272:            self.nlp = spacy.load(language)


=== requirements 파일들에서 'spacy' 언급 검색 ===
/content/MusiConGen/requirements.txt:10:spacy==3.6.1


=== 어떤 패키지가 spacy를 요구하는지 (설치된 패키지 기준) ===
Name: spacy
Version: 3.8.16
Summary: Industrial-strength Natural Language Processing (NLP) in Python
Home-page: https://spacy.io
Author: Explosion
Author-email: contact@explosion.ai
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: catalogue, click, confection, cymem, jinja2, murmurhash, numpy, packaging, preshed, pydantic, requests, setuptools, spacy-legacy, s

In [25]:
# =========================================================
# transformers 실제 사용처 + 현재 설치 상태 확인
# =========================================================

import subprocess

print("=== 코드에서 'transformers' 직접 import 사용처 검색 ===")
result = subprocess.run(
    ["grep", "-rn", "transformers", "/content/MusiConGen", "--include=*.py"],
    capture_output=True, text=True
)
print(result.stdout if result.stdout else "(직접 사용 없음)")

print("\n=== requirements.txt에서 transformers 버전 요구 확인 ===")
result2 = subprocess.run(
    ["grep", "-n", "-i", "transformers", "/content/MusiConGen/requirements.txt"],
    capture_output=True, text=True
)
print(result2.stdout)

print("\n=== 현재 설치된 transformers 버전 (Colab 기본) ===")
result3 = subprocess.run(["pip", "show", "transformers"], capture_output=True, text=True)
print(result3.stdout if result3.stdout else "(설치 안 됨)")

=== 코드에서 'transformers' 직접 import 사용처 검색 ===
/content/MusiConGen/audiocraft/audiocraft/models/encodec.py:19:from transformers import EncodecModel as HFEncodecModel
/content/MusiConGen/audiocraft/audiocraft/metrics/clap_consistency.py:12:from transformers import RobertaTokenizer  # type: ignore
/content/MusiConGen/audiocraft/audiocraft/modules/conditioners.py:22:from transformers import RobertaTokenizer, T5EncoderModel, T5Tokenizer  # type: ignore


=== requirements.txt에서 transformers 버전 요구 확인 ===
14:transformers==4.31.0  # need Encodec there.


=== 현재 설치된 transformers 버전 (Colab 기본) ===
Name: transformers
Version: 5.16.1
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/

In [26]:
from transformers import EncodecModel as HFEncodecModel   # encodec.py
from transformers import RobertaTokenizer                  # clap_consistency.py
from transformers import RobertaTokenizer, T5EncoderModel, T5Tokenizer  # conditioners.py

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [27]:
# numpy 최신으로 되돌려서 단계적으로 어디까지 되는지 확인하기
# =========================================================
# numpy를 Colab 기본(최신) 버전으로 복구
# torch/transformers/jax 등은 이미 numpy 2.x에 맞춰 컴파일된 상태이므로
# numpy만 최신으로 되돌려서 ABI 불일치를 해소한다
# =========================================================

import subprocess, sys

# 1. 현재 numpy 버전 확인
result = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                          capture_output=True, text=True)
print("현재 numpy 버전:", result.stdout.strip(), result.stderr.strip())

# 2. numpy를 강제 재설치 (Colab 기본 최신 버전으로 복구)
#    --force-reinstall로 1.26.4 잔여 파일까지 확실히 덮어씀
print("\n=== numpy 최신 버전으로 재설치 ===")
result2 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "-q", "numpy"],
    capture_output=True, text=True
)
print(result2.stdout[-1500:])
print(result2.stderr[-1500:])

print("\n반드시 런타임을 재시작해야 반영됩니다 (Colab 상단 메뉴: 런타임 > 세션 다시 시작).")
print("재시작 후에는 체크포인트/저장소 파일은 그대로 남아있으니 다운로드를 다시 할 필요는 없습니다.")

현재 numpy 버전: 1.26.4 

=== numpy 최신 버전으로 재설치 ===

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.


반드시 런타임을 재시작해야 반영됩니다 (Colab 상단 메뉴: 런타임 > 세션 다시 시작).
재시작 후에는 체크포인트/저장소 파일은 그대로 남아있으니 다운로드를 다시 할 필요는 없습니다.


In [3]:
import sys

sys.path.insert(0, "/content/MusiConGen/audiocraft")
import audiocraft
print("audiocraft 위치 확인:", audiocraft.__file__)

ModuleNotFoundError: No module named 'xformers'

In [4]:
# =========================================================
# xformers 설치 전, 현재 torch/CUDA 버전 확인
# (xformers는 torch/CUDA 버전과 정확히 맞아야 하는 바이너리 패키지)
# =========================================================

import torch

print("torch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
print("torch가 빌드된 CUDA 버전:", torch.version.cuda)

import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print("\n=== nvidia-smi ===")
print(result.stdout[:500])

torch 버전: 2.11.0+cu128
CUDA 사용 가능: True
torch가 빌드된 CUDA 버전: 12.8

=== nvidia-smi ===
Mon Sep 21 06:53:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       


In [5]:
# =========================================================
# xformers 설치
# - 현재 torch 2.11.0+cu128 확인됨
# - xformers 0.0.35(최신)가 "PyTorch 2.10.0 이후" 사전 빌드 휠 제공한다고 공식 명시
# - 버전 고정 없이 최신 설치 시 pip가 현재 torch에 맞는 휠을 자동 선택
# =========================================================

import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "xformers"],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print(result.stdout[-2000:])
print(result.stderr[-2000:])

# 설치 후 바로 import 검증 (소스 빌드로 빠지지 않고 바이너리 휠이 깔렸는지 확인)
print("\n=== import 검증 ===")
try:
    import xformers
    print("xformers 버전:", xformers.__version__)
    from xformers import ops
    print("xformers.ops import 성공")
except Exception as e:
    print("xformers import 실패:", repr(e))

returncode: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.9 MB/s eta 0:00:00



=== import 검증 ===
xformers 버전: 0.0.35
xformers.ops import 성공


In [8]:
import sys

for mod in list(sys.modules):
    if mod == "audiocraft" or mod.startswith("audiocraft."):
        del sys.modules[mod]

sys.path.insert(0, "/content/MusiConGen/audiocraft/")
import audiocraft
print("audiocraft 위치 확인:", audiocraft.__file__)

ModuleNotFoundError: No module named 'pretty_midi'

In [9]:
# =========================================================
# pretty_midi 설치 + audiocraft가 이후에 또 요구할 수 있는
# requirements.txt 상의 나머지 패키지를 한 번에 미리 점검
# =========================================================

import subprocess, sys

# 1. requirements.txt 전체를 다시 확인해서 우리가 놓친 패키지가 더 있는지 점검
print("=== requirements.txt 전체 목록 ===")
result = subprocess.run(["cat", "/content/MusiConGen/requirements.txt"], capture_output=True, text=True)
print(result.stdout)

# 2. pretty_midi 설치
print("\n=== pretty_midi 설치 ===")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pretty_midi"],
                    capture_output=True, text=True)
print("returncode:", r.returncode)
if r.returncode != 0:
    print(r.stdout[-1500:])
    print(r.stderr[-1500:])

try:
    import pretty_midi
    print("pretty_midi import 성공, 버전:", pretty_midi.__version__)
except Exception as e:
    print("pretty_midi import 실패:", repr(e))

=== requirements.txt 전체 목록 ===
av==11.0.0
einops
flashy==0.0.1
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
torch==2.0.0
torchaudio==2.0.0
tqdm
transformers==4.31.0  # need Encodec there.
xformers==0.0.22
demucs
librosa
soundfile
torchmetrics
encodec
protobuf
torchvision==0.16.0
torchtext==0.16.0
pesq
pystoi

=== pretty_midi 설치 ===
returncode: 0
pretty_midi import 성공, 버전: 0.2.11.post0


In [10]:
import sys

for mod in list(sys.modules):
    if mod == "audiocraft" or mod.startswith("audiocraft."):
        del sys.modules[mod]

sys.path.insert(0, "/content/MusiConGen/audiocraft")
import audiocraft
print("audiocraft 위치 확인:", audiocraft.__file__)

ImportError: Numba needs NumPy 2.2 or less. Got NumPy 2.5.

In [11]:
# =========================================================
# numba를 최신 버전으로 업그레이드
# (0.61.2는 numpy<=2.2만 지원, 우리는 numpy 2.5.3을 쓰고 있음)
# =========================================================

import subprocess, sys

print("=== 현재 numba 버전 확인 ===")
r0 = subprocess.run([sys.executable, "-m", "pip", "show", "numba"], capture_output=True, text=True)
print(r0.stdout)

print("\n=== numba 최신 버전으로 업그레이드 ===")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "-q", "numba"],
    capture_output=True, text=True
)
print("returncode:", r.returncode)
print(r.stdout[-2000:])
print(r.stderr[-2000:])

print("\n=== 업그레이드 후 numba import + numpy 버전 확인 ===")
try:
    import numba
    print("numba 버전:", numba.__version__)
    import numpy
    print("numpy 버전:", numpy.__version__)
except Exception as e:
    print("import 실패:", repr(e))

=== 현재 numba 버전 확인 ===
Name: numba
Version: 0.61.2
Summary: compiling Python code using LLVM
Home-page: https://numba.pydata.org
Author: 
Author-email: 
License: BSD
Location: /usr/local/lib/python3.13/dist-packages
Requires: llvmlite, numpy
Required-by: cudf-cu12, cuml-cu12, librosa, numba-cuda, pynndescent, pytensor, quantecon, segregation, shap, stumpy, umap-learn


=== numba 최신 버전으로 업그레이드 ===
returncode: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 15.3 MB/s eta 0:00:00

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.67.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.67.0 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you hav

In [12]:
import sys

for mod in list(sys.modules):
    if mod == "audiocraft" or mod.startswith("audiocraft.") or mod == "librosa" or mod.startswith("librosa.") or mod == "numba" or mod.startswith("numba."):
        del sys.modules[mod]

sys.path.insert(0, "/content/MusiConGen/audiocraft")
import audiocraft
print("audiocraft 위치 확인:", audiocraft.__file__)

audiocraft 위치 확인: /content/MusiConGen/audiocraft/audiocraft/__init__.py


In [13]:
from transformers import EncodecModel as HFEncodecModel, RobertaTokenizer, T5EncoderModel, T5Tokenizer
print("transformers 관련 클래스 import 성공")

from audiocraft.models import MusicGen
ckpt_dir = "/content/musicongen_checkpoints"
model = MusicGen.get_pretrained(ckpt_dir)
print("모델 로드 완료")

transformers 관련 클래스 import 성공
==== use in-attention: True ====


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

...

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  236MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/116 [00:00<?, ?it/s]

모델 로드 완료


In [14]:
# requirements.txt 업데이트 전 사전확인
# =========================================================
# torchtext, hydra 실제 사용처 확인
# =========================================================

import subprocess

print("=== 코드에서 'torchtext' 사용처 검색 ===")
result = subprocess.run(
    ["grep", "-rn", "torchtext", "/content/MusiConGen", "--include=*.py"],
    capture_output=True, text=True
)
print(result.stdout if result.stdout else "(직접 사용 없음)")

print("\n=== 코드에서 'hydra' 사용처 검색 ===")
result2 = subprocess.run(
    ["grep", "-rln", "hydra", "/content/MusiConGen", "--include=*.py"],
    capture_output=True, text=True
)
print(result2.stdout if result2.stdout else "(직접 사용 없음)")

print("\n=== hydra 사용 파일 개수 및 대표 예시 (상위 10줄) ===")
result3 = subprocess.run(
    ["grep", "-rn", "hydra", "/content/MusiConGen", "--include=*.py"],
    capture_output=True, text=True
)
lines = result3.stdout.splitlines()
print(f"총 매칭 줄 수: {len(lines)}")
for line in lines[:10]:
    print(line)

print("\n=== 지금 audiocraft import 경로(오늘 통과한 경로)에서 hydra/torchtext가 실제로 로드됐는지 ===")
result4 = subprocess.run(
    ["python3", "-c", "import sys; print([m for m in sys.modules if 'hydra' in m or 'torchtext' in m])"],
    capture_output=True, text=True
)
print(result4.stdout, result4.stderr)

=== 코드에서 'torchtext' 사용처 검색 ===
(직접 사용 없음)

=== 코드에서 'hydra' 사용처 검색 ===
/content/MusiConGen/audiocraft/audiocraft/solvers/musicgen.py
/content/MusiConGen/audiocraft/audiocraft/models/loaders.py
/content/MusiConGen/audiocraft/audiocraft/train.py


=== hydra 사용 파일 개수 및 대표 예시 (상위 10줄) ===
총 매칭 줄 수: 12
/content/MusiConGen/audiocraft/audiocraft/solvers/musicgen.py:517:        def get_hydrated_conditions(meta: tp.List[SegmentWithAttributes]):
/content/MusiConGen/audiocraft/audiocraft/solvers/musicgen.py:518:            hydrated_conditions = []
/content/MusiConGen/audiocraft/audiocraft/solvers/musicgen.py:536:                hydrated_conditions.append(cond_dict)
/content/MusiConGen/audiocraft/audiocraft/solvers/musicgen.py:537:            return hydrated_conditions
/content/MusiConGen/audiocraft/audiocraft/solvers/musicgen.py:544:            hydrated_conditions = get_hydrated_conditions(meta)
/content/MusiConGen/audiocraft/audiocraft/solvers/musicgen.py:563:                    gen_unprompted_

In [15]:
# =========================================================
# requirements.txt를 오늘까지 검증된 조합으로 업데이트
# - 원본을 완전히 덮어쓰지 않고, 백업 후 새 파일로 교체
# - 실제로 whack-a-mole을 거쳐 통과가 확인된 버전 정책 반영
# =========================================================

import shutil, os

REPO_REQ = "/content/MusiConGen/requirements.txt"
BACKUP = "/content/MusiConGen/requirements.txt.orig_backup"

# 1. 원본 백업 (되돌릴 수 있게)
if os.path.exists(REPO_REQ) and not os.path.exists(BACKUP):
    shutil.copy(REPO_REQ, BACKUP)
    print(f"원본 백업 완료: {BACKUP}")
else:
    print("백업 파일이 이미 있거나 원본이 없음 - 확인 필요")

# 2. 검증된 새 requirements.txt 내용
new_requirements = """\
# =========================================================
# MusiConGen requirements.txt - Colab (Python 3.13, torch 2.11+cu128) 검증판
# 2026-09-21 트러블슈팅으로 확정. 원본은 requirements.txt.orig_backup 참고.
#
# 정책:
#   - Colab에 이미 설치된 최신 torch/transformers/numpy 스택은 그대로 사용하고
#     버전을 낮추지 않는다 (낮추면 ABI 불일치로 오히려 더 크게 깨짐).
#   - 원본이 옛 버전에 고정했던 패키지 중, 실제로는 정확한 버전이 아니어도
#     되는 것들은 버전 고정을 제거한다.
#   - 소스 빌드가 필요한 패키지(Rust/Cython 컴파일 필요)는 최신 사전빌드
#     휠이 있는 버전으로 대체한다.
# =========================================================

# --- 이미 Colab에 설치되어 있어 버전 고정을 제거한 것들 ---
# spacy: 원본 3.6.1은 blis가 최신 numpy/Cython과 충돌. Colab 기본 3.8.16이면 충분
#        (audiocraft의 conditioners.py는 spacy.load()만 사용, 버전 민감 API 아님)
spacy

# transformers: 원본 4.31.0은 tokenizers(Rust) 소스 빌드 실패 (Rust 컴파일러 없음).
#               Colab 기본 5.x로 EncodecModel/RobertaTokenizer/T5EncoderModel/T5Tokenizer 검증 완료
transformers

# --- torch 계열: Colab 기본 버전 그대로 사용, 버전 고정하지 않음 ---
# 원본 torch==2.0.0 / torchaudio==2.0.0 / torchvision==0.16.0은
# Python 3.13용 wheel 자체가 없어 설치 불가 (pip index versions torch 확인 결과 최저 2.5.0)
torch
torchaudio

# torchtext: PyTorch 공식 deprecated (pytorch/text RFC 확인) + grep으로 audiocraft
#            코드 전체에서 실제 사용처 0건 확인 (2026-09-21) -> 완전 삭제
# (원본에 torchvision==0.16.0과 함께 있었으나 완전히 제거함)

# xformers: torch 버전에 정확히 맞는 사전빌드 휠이 필요.
#           버전 고정 없이 설치하면 pip가 현재 torch(2.11+)에 맞는 최신(0.0.35+)을 자동 선택
xformers

# numpy: 원본 1.24.4 고정은 torch/transformers 최신 스택과 ABI 충돌 (dtype size changed 에러).
#        버전 고정 제거, Colab 기본 최신(2.x) 유지
numpy

# numba: requirements.txt에 명시는 없었지만 librosa의 하위 의존성으로 로드됨.
#        구버전(0.61.2)은 numpy<2.3 상한이 있어 numpy 2.x와 충돌 -> 최신 버전 명시
numba>=0.67.0

# hydra-core: grep 결과 실제 사용처는 audiocraft/train.py 한 곳뿐 (import hydra).
#             musicgen.py의 "hydrated_conditions" 등은 hydra-core와 무관한 변수명(오탐).
#             오늘 검증한 추론 경로(import audiocraft -> MusicGen.get_pretrained)는
#             sys.modules 확인 결과 hydra를 전혀 로드하지 않음 -> 추론에는 불필요.
#             train.py로 재학습/파인튜닝을 시도할 계획이 있을 때만 설치.
#             원본 1.1은 Python 3.13과 충돌 확인됨 (별도 conda 환경 트러블슈팅) -> 버전 고정 제거.
# hydra-core       # 학습(train.py) 필요 시에만 주석 해제
# hydra_colorlog   # 학습(train.py) 필요 시에만 주석 해제

# flashy: 원본 0.0.1 -> 별도 conda 환경에서 0.0.2로 검증됨
flashy

# --- 버전 고정 유지 (원본 그대로, 문제 없음 확인) ---
av
julius
einops
num2words
sentencepiece
tqdm
demucs
librosa
soundfile
torchmetrics
encodec
protobuf
pesq
pystoi

# --- 원본에 누락되어 있었으나 코드에서 실제로 import하는 것 (직접 추가) ---
# audiocraft/modules/conditioners.py: import pretty_midi
pretty_midi
"""

with open(REPO_REQ, "w") as f:
    f.write(new_requirements)

print(f"\n{REPO_REQ} 업데이트 완료")
print("변경 요약:")
print("  - spacy, transformers: 버전 고정 제거 (이미 설치된 최신판 사용)")
print("  - torch, torchaudio, xformers, numpy: 버전 고정 제거 (Colab 기본/최신 사용)")
print("  - torchtext: 제거 (deprecated + grep으로 미사용 확인됨)")
print("  - hydra-core, hydra_colorlog: 주석 처리 (추론 경로엔 불필요, train.py 재학습 시에만 필요)")
print("  - numba: 최신 버전 명시 추가 (원본에 없었음, librosa 하위 의존성)")
print("  - pretty_midi: 추가 (원본에 없었으나 코드에서 실제 사용)")

원본 백업 완료: /content/MusiConGen/requirements.txt.orig_backup

/content/MusiConGen/requirements.txt 업데이트 완료
변경 요약:
  - spacy, transformers: 버전 고정 제거 (이미 설치된 최신판 사용)
  - torch, torchaudio, xformers, numpy: 버전 고정 제거 (Colab 기본/최신 사용)
  - torchtext: 제거 (deprecated + grep으로 미사용 확인됨)
  - hydra-core, hydra_colorlog: 주석 처리 (추론 경로엔 불필요, train.py 재학습 시에만 필요)
  - numba: 최신 버전 명시 추가 (원본에 없었음, librosa 하위 의존성)
  - pretty_midi: 추가 (원본에 없었으나 코드에서 실제 사용)


In [16]:
# musiConGen 실제 오디오 생성 확인
# =========================================================
# 최소 단위 검증: MusicGen 모델이 실제로 generate()를 통해
# 에러 없이 오디오를 출력하는지만 확인 (BPM/코드 조건화 없음)
# =========================================================

import time

# 1. 생성 파라미터를 짧게 설정 (검증 목적이므로 5초만)
model.set_generation_params(duration=5)

# 2. 아주 단순한 텍스트 프롬프트로 생성 시도
prompt = ["a simple acoustic guitar melody, upbeat"]

print("생성 시작...")
start = time.time()

try:
    wav = model.generate(prompt)
    elapsed = time.time() - start
    print(f"생성 성공! 소요 시간: {elapsed:.1f}초")
    print("출력 텐서 shape:", wav.shape)
    print("출력 텐서 dtype:", wav.dtype)
    print("샘플레이트:", model.sample_rate)
except Exception as e:
    print("생성 실패:", repr(e))
    raise

# 3. 실제 오디오 파일로 저장해서 결과물을 눈(귀)으로 확인 가능하게
from audiocraft.data.audio import audio_write

for idx, one_wav in enumerate(wav):
    audio_write(
        f"/content/test_output_{idx}",
        one_wav.cpu(),
        model.sample_rate,
        strategy="loudness",
        loudness_compressor=True,
    )
    print(f"저장 완료: /content/test_output_{idx}.wav")

생성 시작...
생성 성공! 소요 시간: 12.8초
출력 텐서 shape: torch.Size([1, 1, 160000])
출력 텐서 dtype: torch.float32
샘플레이트: 32000
저장 완료: /content/test_output_0.wav


/usr/local/lib/python3.13/dist-packages/torchaudio/__init__.py:178: UserWarning: The 'encoding' parameter is not fully supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(
/usr/local/lib/python3.13/dist-packages/torchaudio/__init__.py:178: UserWarning: The 'bits_per_sample' parameter is not directly supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(


In [17]:
# =========================================================
# 생성된 오디오를 Colab 셀 안에서 바로 재생
# (다운로드 없이 노트북 출력에 오디오 플레이어를 띄움)
# =========================================================

from IPython.display import Audio, display

path = "/content/test_output_0.wav"

print(f"재생할 파일: {path}")
display(Audio(path))

재생할 파일: /content/test_output_0.wav


In [18]:
# 음질 개선
# =========================================================
# 음질 저하 원인 진단
# 1) 체크포인트 파일 실제 크기 확인 (손상/포인터 파일 의심 해소)
# 2) 다른 저장 strategy로 재저장해서 비교
# 3) 파형 기본 통계(클리핑 여부) 확인
# =========================================================

import os

# 1. 체크포인트 파일 실제 크기 확인
ckpt_dir = "/content/musicongen_checkpoints"
for fname in ["compression_state_dict.bin", "state_dict.bin"]:
    path = os.path.join(ckpt_dir, fname)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{fname}: {size_mb:.2f} MB")
    else:
        print(f"{fname}: 파일 없음")

# 2. 파형 기본 통계로 클리핑/이상치 확인
import torch

print("\n=== 생성된 wav 텐서 통계 ===")
print("최댓값:", wav.max().item())
print("최솟값:", wav.min().item())
print("절댓값 1.0 이상 비율(클리핑 의심):", (wav.abs() >= 1.0).float().mean().item())
print("평균 절댓값(RMS 근사):", wav.abs().mean().item())

# 3. loudness normalization 없이 원본 그대로 저장해서 비교
from audiocraft.data.audio import audio_write

for idx, one_wav in enumerate(wav):
    audio_write(
        f"/content/test_output_raw_{idx}",
        one_wav.cpu(),
        model.sample_rate,
        strategy="peak",  # loudness 대신 peak normalization으로 비교
    )
    print(f"저장 완료 (peak 전략): /content/test_output_raw_{idx}.wav")

compression_state_dict.bin: 0.00 MB
state_dict.bin: 2643.00 MB

=== 생성된 wav 텐서 통계 ===
최댓값: 0.9007951617240906
최솟값: -1.1247299909591675
절댓값 1.0 이상 비율(클리핑 의심): 0.00028124998789280653
평균 절댓값(RMS 근사): 0.10040605813264847
저장 완료 (peak 전략): /content/test_output_raw_0.wav


In [19]:
from IPython.display import Audio, display

path = "/content/test_output_raw_0.wav"

print(f"재생할 파일: {path}")
display(Audio(path))

재생할 파일: /content/test_output_raw_0.wav


In [20]:
# =========================================================
# 실제로 model 객체가 어떤 compression_model(Encodec)을 쓰고 있는지 확인
# - 로컬의 빈 compression_state_dict.bin이 정말 문제였는지
# - 폴백으로 다른 소스에서 로드됐는지 확인
# =========================================================

print("=== model.compression_model 정보 ===")
print("타입:", type(model.compression_model))

# 원본 체크포인트 파일 자체를 다시 검사 (진짜 0바이트인지, LFS 포인터 텍스트인지)
import os
path = "/content/musicongen_checkpoints/compression_state_dict.bin"
size = os.path.getsize(path)
print(f"\ncompression_state_dict.bin 실제 크기: {size} bytes")

with open(path, "rb") as f:
    head = f.read(200)
print("파일 시작 부분 (텍스트로 디코딩 시도):")
try:
    print(head.decode("utf-8"))
except Exception:
    print(head)

# model 로드 시 어떤 huggingface 경로가 실제로 사용됐는지 확인 가능한 속성 탐색
print("\n=== model 객체 주요 속성 ===")
for attr in ["name", "compression_model"]:
    if hasattr(model, attr):
        val = getattr(model, attr)
        print(f"{attr}: {val if not hasattr(val, '__class__') else val.__class__}")

=== model.compression_model 정보 ===
타입: <class 'audiocraft.models.encodec.HFEncodecCompressionModel'>

compression_state_dict.bin 실제 크기: 589 bytes
파일 시작 부분 (텍스트로 디코딩 시도):
b'PK\x03\x04\x00\x00\x08\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x1f\x00C\x00compression_state_dict/data.pklFB?\x00ZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZZ\x80\x02}q\x00(X\n\x00\x00\x00pretrainedq\x01X\x16\x00\x00\x00facebook/encodec_32khzq\x02X\x08\x00\x00\x00exportedq\x03\x88X\x07\x00\x00'

=== model 객체 주요 속성 ===
name: <class 'str'>
compression_model: <class 'audiocraft.models.encodec.HFEncodecCompressionModel'>


In [21]:
# =========================================================
# Drive 마운트 + 기존에 저장된 Demucs 결과물 존재 확인
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

import os

# stem-remix-assistant 관련 폴더 구조 확인
base = "/content/drive/MyDrive/stem-remix-assistant"
print(f"=== {base} 존재 여부: {os.path.exists(base)} ===")
if os.path.exists(base):
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/")
        for f in files:
            print(f"{indent}  {f}")

# demucs, htdemucs 등 관련 키워드로 Drive 전체에서 검색 (너무 오래 걸리면 중단)
print("\n=== MyDrive 전체에서 'demucs' 포함 경로 검색 (상위 폴더만) ===")
mydrive = "/content/drive/MyDrive"
for item in os.listdir(mydrive):
    if "demucs" in item.lower() or "stem" in item.lower():
        print(os.path.join(mydrive, item))

Mounted at /content/drive
=== /content/drive/MyDrive/stem-remix-assistant 존재 여부: True ===
stem-remix-assistant/
  Copy of 01_setup_and_test.ipynb
  requirements_filtered.txt
  stemremix_installed_versions.txt
  outputs/
    draft_0.wav
    separated/
      htdemucs/
        draft_0/
          drums.wav
          bass.wav
          other.wav
          vocals.wav

=== MyDrive 전체에서 'demucs' 포함 경로 검색 (상위 폴더만) ===
/content/drive/MyDrive/stem-remix-assistant


In [22]:
# =========================================================
# 오늘 생성한 MusicGen 결과물을 Drive의 기존 구조에 맞춰 저장
# 기존 구조: stem-remix-assistant/outputs/separated/htdemucs/draft_0/*.wav
# 신규 추가: stem-remix-assistant/outputs/musicongen_generated/*.wav
# =========================================================

import os, shutil

DRIVE_BASE = "/content/drive/MyDrive/stem-remix-assistant/outputs"
SAVE_DIR = os.path.join(DRIVE_BASE, "musicongen_generated")
os.makedirs(SAVE_DIR, exist_ok=True)

# 오늘 로컬(/content)에 생성해둔 결과물들
local_files = [
    "/content/test_output_0.wav",       # loudness 정규화 버전
    "/content/test_output_raw_0.wav",   # peak 정규화 버전 (비교용)
]

saved = []
for src in local_files:
    if os.path.exists(src):
        dst = os.path.join(SAVE_DIR, os.path.basename(src))
        shutil.copy(src, dst)
        saved.append(dst)
        print(f"저장 완료: {dst}")
    else:
        print(f"파일 없음 (건너뜀): {src}")

print(f"\n총 {len(saved)}개 파일을 Drive에 저장했습니다.")
print(f"저장 위치: {SAVE_DIR}")

# 확인
print("\n=== 저장 폴더 내용 확인 ===")
for f in sorted(os.listdir(SAVE_DIR)):
    full = os.path.join(SAVE_DIR, f)
    size_kb = os.path.getsize(full) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

저장 완료: /content/drive/MyDrive/stem-remix-assistant/outputs/musicongen_generated/test_output_0.wav
저장 완료: /content/drive/MyDrive/stem-remix-assistant/outputs/musicongen_generated/test_output_raw_0.wav

총 2개 파일을 Drive에 저장했습니다.
저장 위치: /content/drive/MyDrive/stem-remix-assistant/outputs/musicongen_generated

=== 저장 폴더 내용 확인 ===
  test_output_0.wav (312.6 KB)
  test_output_raw_0.wav (312.6 KB)


## 3. 분석 도구 설치 (madmom) + config.yaml 로드

In [23]:
# 경로 확인 후 수정
# =========================================================
# madmom 분석 단계 진입 전, 실제 존재하는 경로들을 전부 확인
# (노트북 원본의 상대경로 가정이 지금 환경과 안 맞을 가능성 검증)
# =========================================================

import os

print("현재 작업 디렉토리:", os.getcwd())

# 1. GitHub에서 clone된 stem-remix-assistant 저장소가 있는지 확인
candidates_repo = [
    "/content/stem-remix-assistant",
    "/content/MusiConGen/../stem-remix-assistant",
]
print("\n=== stem-remix-assistant 저장소 clone 여부 ===")
for c in candidates_repo:
    real = os.path.realpath(c)
    print(f"{c} (실제: {real}) 존재? {os.path.exists(real)}")

# 2. config.yaml 후보 경로들
print("\n=== config.yaml 후보 경로 ===")
config_candidates = [
    "/content/stem-remix-assistant/experiments/exp_002_musicongen/config.yaml",
]
for c in config_candidates:
    print(f"{c} 존재? {os.path.exists(c)}")

# 3. 원곡/샘플 파일 후보 경로들 (docs/samples vs Drive의 outputs)
print("\n=== 원곡 및 스템 샘플 후보 경로 ===")
sample_candidates = [
    "/content/stem-remix-assistant/docs/samples/draft_0.wav",
    "/content/drive/MyDrive/stem-remix-assistant/outputs/draft_0.wav",
    "/content/drive/MyDrive/stem-remix-assistant/outputs/separated/htdemucs/draft_0/drums.wav",
    "/content/drive/MyDrive/stem-remix-assistant/outputs/separated/htdemucs/draft_0/bass.wav",
    "/content/drive/MyDrive/stem-remix-assistant/outputs/separated/htdemucs/draft_0/other.wav",
    "/content/drive/MyDrive/stem-remix-assistant/outputs/separated/htdemucs/draft_0/vocals.wav",
]
for c in sample_candidates:
    print(f"{c} 존재? {os.path.exists(c)}")

# 4. 체크포인트 경로 재확인 (오늘 우리가 실제로 쓴 경로)
print("\n=== 체크포인트 경로 ===")
ckpt_candidates = [
    "/content/musicongen_checkpoints",
    "/content/MusiConGen/ckpt/musicongen",
]
for c in ckpt_candidates:
    print(f"{c} 존재? {os.path.exists(c)}")
    if os.path.exists(c):
        print("  내용:", os.listdir(c))

현재 작업 디렉토리: /content

=== stem-remix-assistant 저장소 clone 여부 ===
/content/stem-remix-assistant (실제: /content/stem-remix-assistant) 존재? False
/content/MusiConGen/../stem-remix-assistant (실제: /content/stem-remix-assistant) 존재? False

=== config.yaml 후보 경로 ===
/content/stem-remix-assistant/experiments/exp_002_musicongen/config.yaml 존재? False

=== 원곡 및 스템 샘플 후보 경로 ===
/content/stem-remix-assistant/docs/samples/draft_0.wav 존재? False
/content/drive/MyDrive/stem-remix-assistant/outputs/draft_0.wav 존재? True
/content/drive/MyDrive/stem-remix-assistant/outputs/separated/htdemucs/draft_0/drums.wav 존재? True
/content/drive/MyDrive/stem-remix-assistant/outputs/separated/htdemucs/draft_0/bass.wav 존재? True
/content/drive/MyDrive/stem-remix-assistant/outputs/separated/htdemucs/draft_0/other.wav 존재? True
/content/drive/MyDrive/stem-remix-assistant/outputs/separated/htdemucs/draft_0/vocals.wav 존재? True

=== 체크포인트 경로 ===
/content/musicongen_checkpoints 존재? True
  내용: ['state_dict.bin', '.cache', 'compressi

In [24]:
# =========================================================
# 3단계: madmom 설치 + config.yaml 로드 (경로 수정판)
# - GitHub 저장소를 clone하지 않고, config.yaml만 직접 다운로드
# - 오디오 샘플은 실제 위치인 Drive 경로를 사용 (docs/samples 아님)
# =========================================================

!pip install madmom pyyaml librosa soundfile -q

import os, yaml

# config.yaml을 GitHub에서 직접 받아옴 (저장소 전체를 clone할 필요 없음)
os.makedirs("/content/exp_002_musicongen", exist_ok=True)
CONFIG_PATH = "/content/exp_002_musicongen/config.yaml"

!wget -q -O {CONFIG_PATH} "https://raw.githubusercontent.com/finneKIM/stem-remix-assistant/main/experiments/exp_002_musicongen/config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

TARGET_STEM = cfg["target_stem"]
PROMPT = cfg["prompt"]
DURATION_SWEEP = cfg["duration_sec_sweep"]
SEED = cfg["seed"]

print("target_stem:", TARGET_STEM)
print("prompt:", PROMPT)
print("duration_sec_sweep:", DURATION_SWEEP)
print("seed:", SEED)

# 실제 존재를 확인한 Drive 경로로 원곡/스템 경로 지정
# (원본 노트북의 '../stem-remix-assistant/docs/samples/...' 가정은 틀림 - 그 경로엔 파일 없음)
DRIVE_OUTPUTS = "/content/drive/MyDrive/stem-remix-assistant/outputs"
ORIG_TRACK = f"{DRIVE_OUTPUTS}/draft_0.wav"
ORIG_STEM = f"{DRIVE_OUTPUTS}/separated/htdemucs/draft_0/{TARGET_STEM}.wav"

print("\n=== 오디오 경로 확인 ===")
print("원곡:", ORIG_TRACK, "존재?", os.path.exists(ORIG_TRACK))
print("원곡 스템:", ORIG_STEM, "존재?", os.path.exists(ORIG_STEM))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 70.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
target_stem: drums
prompt: punchier trap-style drums, same tempo
duration_sec_sweep: [10, 15, 20]
seed: 42

=== 오디오 경로 확인 ===
원곡: /content/drive/MyDrive/stem-remix-assistant/outputs/draft_0.wav 존재? True
원곡 스템: /content/drive/MyDrive/stem-remix-assistant/outputs/separated/htdemucs/draft_0/drums.wav 존재? True


## 4. 원곡·원곡 스템에서 BPM/코드/비트 추출

EXP-001에서 만든 `docs/samples/draft_0.wav`(원곡)와 `docs/samples/{target_stem}.wav`(원곡 스템)를 기준으로 madmom 분석. 여기서 얻은 BPM/코드가 MusiConGen 조건화 입력이자, 나중에 재생성 스템과 비교할 기준값.

In [25]:
# =========================================================
# 4단계: 원곡/원곡 스템에서 BPM/코드/비트/온셋 추출
# =========================================================

from madmom.features.beats import RNNBeatProcessor, DBNBeatTrackingProcessor
from madmom.features.tempo import TempoEstimationProcessor
from madmom.features.chords import DeepChromaChordRecognitionProcessor
from madmom.features.chroma import DeepChromaProcessor
from madmom.features.onsets import CNNOnsetProcessor, OnsetPeakPickingProcessor
import numpy as np

def get_bpm(path):
    act = RNNBeatProcessor()(path)
    tempo_proc = TempoEstimationProcessor(fps=100)
    tempi = tempo_proc(act)
    return float(tempi[0][0])  # 가장 confidence 높은 BPM 후보

def get_beats(path):
    act = RNNBeatProcessor()(path)
    beat_proc = DBNBeatTrackingProcessor(fps=100)
    return beat_proc(act)

def get_onsets(path):
    act = CNNOnsetProcessor()(path)
    onset_proc = OnsetPeakPickingProcessor(fps=100)
    return onset_proc(act)

def get_chords(path):
    chroma = DeepChromaProcessor()(path)
    chord_proc = DeepChromaChordRecognitionProcessor()
    return chord_proc(chroma)  # [(start, end, chord_label), ...]

print("=== 원곡(draft_0.wav) 분석 시작 ===")
orig_bpm = get_bpm(ORIG_TRACK)
print("원곡 BPM:", orig_bpm)

orig_beats = get_beats(ORIG_TRACK)
print("원곡 비트 개수:", len(orig_beats))

orig_onsets = get_onsets(ORIG_TRACK)
print("원곡 온셋 개수:", len(orig_onsets))

orig_chords = get_chords(ORIG_TRACK)
print("원곡 코드 이벤트 개수:", len(orig_chords))
print("코드 이벤트 예시 (앞 5개):", orig_chords[:5])

print("\n=== 원곡 스템(drums.wav) 분석 시작 ===")
stem_bpm = get_bpm(ORIG_STEM)
print("스템 BPM:", stem_bpm)

stem_beats = get_beats(ORIG_STEM)
print("스템 비트 개수:", len(stem_beats))

print("\n=== 요약 ===")
print(f"원곡 BPM: {orig_bpm:.1f} / 스템(drums) BPM: {stem_bpm:.1f}")
print(f"BPM 차이: {abs(orig_bpm - stem_bpm):.2f}")

/usr/local/lib/python3.13/dist-packages/madmom/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


ImportError: cannot import name 'MutableSequence' from 'collections' (/usr/lib/python3.13/collections/__init__.py)

In [26]:
# =========================================================
# madmom 소스 직접 패치 시도
# collections.MutableSequence -> collections.abc.MutableSequence
# Python 3.10+에서 제거된 옛날 문법을 madmom 코드 전체에서 찾아 고침
# =========================================================

import subprocess, re, os

# 1. madmom 패키지 설치 위치 확인
result = subprocess.run(
    ["python3", "-c", "import madmom, os; print(os.path.dirname(madmom.__file__))"],
    capture_output=True, text=True
)
# madmom import 자체가 실패하므로 pip show로 위치 확인
result = subprocess.run(["pip", "show", "-f", "madmom"], capture_output=True, text=True)
print(result.stdout[:500])

# pip show로 Location 파싱
location = None
for line in result.stdout.splitlines():
    if line.startswith("Location:"):
        location = line.split("Location:")[1].strip()
madmom_dir = os.path.join(location, "madmom") if location else None
print("madmom 패키지 경로:", madmom_dir)

# 2. collections에서 직접 import하는 옛날 문법을 쓰는 파일 전부 검색
print("\n=== 'from collections import' (abc 아닌) 문제 패턴 검색 ===")
search = subprocess.run(
    ["grep", "-rln", "-E",
     r"from collections import (MutableSequence|MutableMapping|MutableSet|Sequence|Mapping|Set|Iterable|Callable)",
     madmom_dir],
    capture_output=True, text=True
)
problem_files = search.stdout.strip().splitlines()
print(f"문제 파일 {len(problem_files)}개 발견:")
for f in problem_files:
    print(" ", f)

# 3. 각 파일에서 실제로 어떤 줄이 문제인지 확인
print("\n=== 각 파일의 문제 줄 ===")
for f in problem_files:
    result = subprocess.run(
        ["grep", "-n", "-E",
         r"from collections import (MutableSequence|MutableMapping|MutableSet|Sequence|Mapping|Set|Iterable|Callable)",
         f],
        capture_output=True, text=True
    )
    print(f"{f}:")
    print(result.stdout)

# 4. 패치 적용 - collections -> collections.abc 로 교체
print("\n=== 패치 적용 ===")
pattern = re.compile(
    r"from collections import (MutableSequence|MutableMapping|MutableSet|Sequence|Mapping|Set|Iterable|Callable)"
)

patched_count = 0
for f in problem_files:
    with open(f, "r") as fh:
        content = fh.read()
    new_content = pattern.sub(r"from collections.abc import \1", content)
    if new_content != content:
        with open(f, "w") as fh:
            fh.write(new_content)
        patched_count += 1
        print(f"패치 완료: {f}")

print(f"\n총 {patched_count}개 파일 패치 완료")

# 5. 재검증
print("\n=== import audiocraft 대신 madmom 단독 재검증 ===")
result = subprocess.run(
    ["python3", "-c",
     "from madmom.features.beats import RNNBeatProcessor, DBNBeatTrackingProcessor; "
     "from madmom.features.tempo import TempoEstimationProcessor; "
     "from madmom.features.chords import DeepChromaChordRecognitionProcessor; "
     "from madmom.features.chroma import DeepChromaProcessor; "
     "from madmom.features.onsets import CNNOnsetProcessor, OnsetPeakPickingProcessor; "
     "print('madmom import 전체 성공')"],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT:", result.stdout)
print("STDERR (마지막 3000자):", result.stderr[-3000:])

Name: madmom
Version: 0.16.1
Summary: Python audio signal processing library
Home-page: https://github.com/CPJKU/madmom
Author: Department of Computational Perception, Johannes Kepler University, Linz, Austria and Austrian Research Institute for Artificial Intelligence (OFAI), Vienna, Austria
Author-email: madmom-users@googlegroups.com
License: BSD, CC BY-NC-SA
Location: /usr/local/lib/python3.13/dist-packages
Requires: cython, mido, numpy, scipy
Required-by: 
Files:
  ../../../bin/BarTracker
  
madmom 패키지 경로: /usr/local/lib/python3.13/dist-packages/madmom

=== 'from collections import' (abc 아닌) 문제 패턴 검색 ===
문제 파일 1개 발견:
  /usr/local/lib/python3.13/dist-packages/madmom/processors.py

=== 각 파일의 문제 줄 ===
/usr/local/lib/python3.13/dist-packages/madmom/processors.py:
23:from collections import MutableSequence


=== 패치 적용 ===
패치 완료: /usr/local/lib/python3.13/dist-packages/madmom/processors.py

총 1개 파일 패치 완료

=== import audiocraft 대신 madmom 단독 재검증 ===
returncode: 1
STDOUT: 
STDERR (마지막 3000자

In [27]:
# madmom 전체스캔

# =========================================================
# madmom 전체에서 numpy 제거된 별칭(np.float, np.int, np.bool 등) 사용처
# 그리고 collections 관련 남은 문제까지 한 번에 스캔
# 하나씩 잡지 않고 전체 규모를 먼저 파악
# =========================================================

import subprocess

madmom_dir = "/usr/local/lib/python3.13/dist-packages/madmom"

# numpy에서 제거된 별칭들 (1.20에서 deprecated, 1.24+에서 완전 제거)
deprecated_np_aliases = ["np.float", "np.int", "np.bool", "np.object", "np.str", "np.complex", "np.long"]

print("=== madmom 전체에서 제거된 numpy 별칭 사용처 스캔 ===\n")
total_files = set()
total_lines = 0

for alias in deprecated_np_aliases:
    # np.float64 같은 정상 케이스는 제외하기 위해 뒤에 알파벳/숫자가 안 오는 경우만 매칭
    pattern = alias.replace(".", r"\.") + r"(?![a-zA-Z0-9_])"
    result = subprocess.run(
        ["grep", "-rnE", pattern, madmom_dir, "--include=*.py"],
        capture_output=True, text=True
    )
    lines = [l for l in result.stdout.splitlines() if l.strip()]
    if lines:
        print(f"--- {alias} : {len(lines)}건 ---")
        for l in lines[:5]:
            print(" ", l)
        if len(lines) > 5:
            print(f"  ... 외 {len(lines)-5}건 더")
        print()
        total_lines += len(lines)
        for l in lines:
            fname = l.split(":")[0]
            total_files.add(fname)

print(f"\n=== 요약 ===")
print(f"영향받는 파일 수: {len(total_files)}")
print(f"영향받는 줄 수: {total_lines}")
print("\n영향받는 파일 목록:")
for f in sorted(total_files):
    print(" ", f)

=== madmom 전체에서 제거된 numpy 별칭 사용처 스캔 ===


=== 요약 ===
영향받는 파일 수: 0
영향받는 줄 수: 0

영향받는 파일 목록:


In [28]:
# =========================================================
# 1) io/__init__.py가 이미 고쳐졌는지 직접 확인
# 2) Python re 모듈로 madmom 전체를 정확하게 재스캔
# =========================================================

import os, re

madmom_dir = "/usr/local/lib/python3.13/dist-packages/madmom"

# 1. io/__init__.py의 문제 줄 상태 직접 확인
print("=== 1. madmom/io/__init__.py 현재 상태 확인 ===")
io_init = os.path.join(madmom_dir, "io", "__init__.py")
with open(io_init) as f:
    lines = f.readlines()

found = False
for i, line in enumerate(lines, start=1):
    if "SEGMENT_DTYPE" in line or re.search(r"np\.(float|int|bool|object|str|complex|long)(?![a-zA-Z0-9_])", line):
        print(f"  {i}: {line.rstrip()}")
        found = True
if not found:
    print("  SEGMENT_DTYPE 또는 np.float류 패턴 없음 (이미 수정됐거나 다른 위치)")

# 2. Python re 모듈로 madmom 전체 정확히 재스캔
print("\n=== 2. Python re 모듈로 전체 재스캔 ===")

deprecated_pattern = re.compile(r"\bnp\.(float|int|bool|object|str|complex|long)\b(?!\w)")
collections_pattern = re.compile(
    r"from collections import .*(MutableSequence|MutableMapping|MutableSet|Sequence|Mapping|Set|Iterable|Callable)"
)

np_hits = []
collections_hits = []

for root, dirs, files in os.walk(madmom_dir):
    for fname in files:
        if not fname.endswith(".py"):
            continue
        fpath = os.path.join(root, fname)
        try:
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                for lineno, line in enumerate(f, start=1):
                    if deprecated_pattern.search(line):
                        np_hits.append((fpath, lineno, line.strip()))
                    if collections_pattern.search(line):
                        collections_hits.append((fpath, lineno, line.strip()))
        except Exception as e:
            print(f"  읽기 실패: {fpath} ({e})")

print(f"\nnp.float 등 제거된 numpy 별칭: {len(np_hits)}건")
for fpath, lineno, content in np_hits[:20]:
    print(f"  {fpath}:{lineno}: {content}")
if len(np_hits) > 20:
    print(f"  ... 외 {len(np_hits)-20}건 더")

print(f"\ncollections 직접 import (abc 아님): {len(collections_hits)}건")
for fpath, lineno, content in collections_hits:
    print(f"  {fpath}:{lineno}: {content}")

print(f"\n=== 최종 요약 ===")
print(f"np.float류 문제: {len(np_hits)}건 / {len(set(h[0] for h in np_hits))}개 파일")
print(f"collections 문제: {len(collections_hits)}건 / {len(set(h[0] for h in collections_hits))}개 파일")

=== 1. madmom/io/__init__.py 현재 상태 확인 ===
  22: SEGMENT_DTYPE = [('start', np.float), ('end', np.float), ('label', object)]
  319:     segments = np.zeros(len(start), dtype=SEGMENT_DTYPE)
  333:         Labelled segments, one per row (column definition see SEGMENT_DTYPE).

=== 2. Python re 모듈로 전체 재스캔 ===

np.float 등 제거된 numpy 별칭: 97건
  /usr/local/lib/python3.13/dist-packages/madmom/audio/signal.py:355: if signal.dtype != np.float:
  /usr/local/lib/python3.13/dist-packages/madmom/audio/signal.py:356: signal = signal.astype(np.float)
  /usr/local/lib/python3.13/dist-packages/madmom/audio/stft.py:273: >>> sig = Signal('tests/data/audio/sample.wav', dtype=np.float)
  /usr/local/lib/python3.13/dist-packages/madmom/audio/filters.py:167: return (12. * np.log2(np.asarray(f, dtype=np.float) / fref)) + 69.
  /usr/local/lib/python3.13/dist-packages/madmom/audio/filters.py:187: return 2. ** ((np.asarray(m, dtype=np.float) - 69.) / 12.) * fref
  /usr/local/lib/python3.13/dist-packages/madmom/audio/

#  madmom → librosa(비트/온셋/다운비트) + BTC-ISMIR2019(코드) 전환

In [29]:
# =========================================================
# BTC-ISMIR2019 검증 1단계: 저장소 clone + 구조/체크포인트 확인
# - madmom처럼 바이너리 호환성 문제가 있는지 먼저 "겉모습"부터 점검
# - 체크포인트가 저장소에 포함되어 있는지, 아니면 별도 다운로드가
#   필요한지는 공식 README에 명시되어 있지 않았으므로 여기서 직접 확인
# =========================================================

import subprocess, os

BTC_DIR = "/content/BTC-ISMIR19"

# 1. 저장소 clone (이미 있으면 스킵)
if not os.path.exists(BTC_DIR):
    result = subprocess.run(
        ["git", "clone", "https://github.com/jayg996/BTC-ISMIR19.git", BTC_DIR],
        capture_output=True, text=True
    )
    print("=== git clone ===")
    print("returncode:", result.returncode)
    print(result.stdout[-1000:])
    print(result.stderr[-1000:])
else:
    print("이미 clone되어 있음:", BTC_DIR)

# 2. 저장소 전체 구조 확인 (체크포인트 파일이 포함돼 있는지 직접 확인)
print("\n=== 저장소 파일 트리 ===")
for root, dirs, files in os.walk(BTC_DIR):
    # .git 폴더는 제외
    dirs[:] = [d for d in dirs if d != ".git"]
    level = root.replace(BTC_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"{indent}  {f} ({size_mb:.2f} MB)")

# 3. requirements 관련 파일이 실제로 있는지 (README에서 못 찾았으므로 재확인)
print("\n=== requirements 관련 파일 존재 여부 ===")
for fname in ["requirements.txt", "environment.yml", "Pipfile", "setup.py"]:
    fpath = os.path.join(BTC_DIR, fname)
    print(f"  {fname}: {'있음' if os.path.exists(fpath) else '없음'}")

# 4. model/ 디렉토리(run_config.yaml의 ckpt_path 기본값) 확인
print("\n=== model/ 디렉토리(체크포인트 저장 위치) 확인 ===")
model_dir = os.path.join(BTC_DIR, "model")
if os.path.exists(model_dir):
    for f in os.listdir(model_dir):
        print(" ", f)
else:
    print("  model/ 디렉토리 없음 -> 체크포인트를 직접 받아서 넣어야 할 가능성 높음")

# 5. test.py 안에서 체크포인트를 자동 다운로드하는 로직이 있는지 확인
print("\n=== test.py 내용에서 다운로드/체크포인트 관련 로직 확인 ===")
test_py = os.path.join(BTC_DIR, "test.py")
if os.path.exists(test_py):
    with open(test_py) as f:
        content = f.read()
    print(content[:3000])
else:
    print("test.py 없음")

=== git clone ===
returncode: 0

Cloning into '/content/BTC-ISMIR19'...


=== 저장소 파일 트리 ===
BTC-ISMIR19/
  baseline_models.py (0.01 MB)
  README.md (0.00 MB)
  run_config.yaml (0.00 MB)
  train.py (0.01 MB)
  audio_dataset.py (0.01 MB)
  crf_model.py (0.01 MB)
  LICENSE (0.00 MB)
  train_crf.py (0.01 MB)
  test.py (0.00 MB)
  btc_model.py (0.01 MB)
  png/
    model.png (0.09 MB)
    attention.png (1.56 MB)
    example.png (0.01 MB)
  utils/
    __init__.py (0.00 MB)
    mir_eval_modules.py (0.03 MB)
    preprocess.py (0.02 MB)
    logger.py (0.00 MB)
    chords.py (0.02 MB)
    hparams.py (0.00 MB)
    tf_logger.py (0.00 MB)
    pytorch_utils.py (0.00 MB)
    transformer_modules.py (0.01 MB)
    __pycache__/
      transformer_modules.cpython-36.pyc (0.01 MB)
      logger.cpython-36.pyc (0.00 MB)
      hparams.cpython-36.pyc (0.00 MB)
      mir_eval_modules.cpython-36.pyc (0.01 MB)
      __init__.cpython-36.pyc (0.00 MB)
  test/
    example.mp3 (9.95 MB)
    btc_model_large_voca.pt (11.

In [30]:
# =========================================================
# BTC-ISMIR2019 검증 2단계: 의존성 설치 + import 검증
# - 1단계(step1) 실행 결과를 보고 나서 진행할 것
#   (체크포인트 위치, requirements.txt 존재 여부가 여기서 갈릴 수 있음)
# - madmom과 달리 순수 PyTorch 기반이라 Cython/C 확장이 없다는 게
#   README 기준 확인된 사실이지만, 실제 설치는 다른 문제이므로 직접 검증
# =========================================================

import subprocess, sys

# README에 명시된 최소 버전 요구사항 (requirements.txt가 없으므로 이걸 기준으로 설치)
# 지금 Colab 환경에 이미 깔려있는 numpy/torch(오늘 MusicGen 세팅에서 검증된 버전)를
# 최대한 건드리지 않고, 부족한 패키지만 추가 설치하는 방식으로 진행
deps_to_check = [
    "torch",
    "numpy",
    "pandas",
    "pyrubberband",
    "librosa",
    "yaml",       # pyyaml
    "mir_eval",
    "pretty_midi",
]

print("=== 설치 전 현재 버전 확인 ===")
for mod in deps_to_check:
    try:
        m = __import__(mod)
        version = getattr(m, "__version__", "버전 정보 없음")
        print(f"  {mod}: 이미 설치됨 (버전 {version})")
    except ImportError:
        print(f"  {mod}: 설치 안 됨")

# pyrubberband는 시스템 바이너리(rubberband-cli)가 필요할 수 있음 -> 별도 체크
print("\n=== rubberband 시스템 바이너리 확인 (pyrubberband의 실제 의존성) ===")
result = subprocess.run(["which", "rubberband"], capture_output=True, text=True)
if result.stdout.strip():
    print("  rubberband CLI 있음:", result.stdout.strip())
else:
    print("  rubberband CLI 없음 -> apt-get install rubberband-cli 필요할 수 있음")

# 부족한 것만 설치 (numpy/torch는 오늘 세팅된 버전을 건드리지 않도록 버전 고정 없이 설치)
print("\n=== 부족한 패키지 설치 ===")
install_targets = ["pandas", "pyrubberband", "mir_eval", "pretty_midi", "pyyaml"]
result = subprocess.run(
    [sys.executable, "-m", "pip", "install"] + install_targets,
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print(result.stdout[-2000:])
print(result.stderr[-2000:])

# apt로 rubberband-cli도 시도 (Colab은 sudo 없이 apt 가능)
print("\n=== rubberband-cli apt 설치 시도 ===")
result = subprocess.run(
    ["apt-get", "install", "-y", "rubberband-cli"],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print(result.stdout[-1000:])
print(result.stderr[-1000:])

# 최종 import 검증 (madmom 때와 동일하게, 여기서 whack-a-mole 패턴이 재현되는지 확인)
print("\n=== 최종 import 검증 ===")
result = subprocess.run(
    [sys.executable, "-c",
     "import torch, numpy, pandas, pyrubberband, librosa, yaml, mir_eval, pretty_midi; "
     "print('BTC-ISMIR2019 의존성 전부 import 성공')"],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT:", result.stdout)
print("STDERR (마지막 3000자):", result.stderr[-3000:])

=== 설치 전 현재 버전 확인 ===
  torch: 이미 설치됨 (버전 2.11.0+cu128)
  numpy: 이미 설치됨 (버전 2.5.3)
  pandas: 이미 설치됨 (버전 2.2.3)
  pyrubberband: 설치 안 됨
  librosa: 이미 설치됨 (버전 0.11.0)
  yaml: 이미 설치됨 (버전 6.0.3)
  mir_eval: 설치 안 됨
  pretty_midi: 이미 설치됨 (버전 0.2.11.post0)

=== rubberband 시스템 바이너리 확인 (pyrubberband의 실제 의존성) ===
  rubberband CLI 없음 -> apt-get install rubberband-cli 필요할 수 있음

=== 부족한 패키지 설치 ===
returncode: 0
13/dist-packages (6.0.3)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 5.3 MB/s eta 0:00:00



=== rubberband-cli apt 설치 시도 ===
returncode: 0
oble/universe amd64 rubberband-cli amd64 3.3.0+dfsg-2build1 [129 kB]
Fetched 129 kB in 0s (1,455 kB/s)
Selecting previously unselected package rubberband-cli.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Re

In [31]:
# =========================================================
# BTC-ISMIR2019 검증 3단계: 실제 추론 테스트
# - step1, step2가 모두 성공한 뒤에 실행
# - 테스트용 오디오는 오늘 Drive에 저장해둔 원곡(draft_0.wav)을 사용
# - 체크포인트는 step1에서 확인한 실제 경로/방법에 맞춰
#   아래 CKPT_PATH를 수정해서 사용할 것 (자동 다운로드가 없다면
#   README/이슈에서 안내하는 방법으로 수동 다운로드 후 경로 지정)
# =========================================================

import subprocess, os, shutil

BTC_DIR = "/content/BTC-ISMIR19"
DRIVE_OUTPUTS = "/content/drive/MyDrive/stem-remix-assistant/outputs"
TEST_AUDIO_SRC = f"{DRIVE_OUTPUTS}/draft_0.wav"

# BTC-ISMIR2019의 test.py는 폴더 단위로 오디오를 읽으므로
# 테스트용 audio_dir을 새로 만들어 원곡 하나만 복사
TEST_AUDIO_DIR = os.path.join(BTC_DIR, "my_test_audio")
SAVE_DIR = os.path.join(BTC_DIR, "my_test_result")

os.makedirs(TEST_AUDIO_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print("=== 원곡 존재 확인 ===")
print(TEST_AUDIO_SRC, "존재?", os.path.exists(TEST_AUDIO_SRC))

if os.path.exists(TEST_AUDIO_SRC):
    dest = os.path.join(TEST_AUDIO_DIR, "draft_0.wav")
    shutil.copy(TEST_AUDIO_SRC, dest)
    print("복사 완료:", dest)

# 실제 추론 실행
# --voca False: major/minor 라벨 타입 (25개 클래스, README 기준 기본값)
print("\n=== BTC-ISMIR2019 추론 실행 ===")
result = subprocess.run(
    ["python", "test.py",
     "--audio_dir", TEST_AUDIO_DIR,
     "--save_dir", SAVE_DIR,
     "--voca", "False"],
    cwd=BTC_DIR,
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT (마지막 3000자):", result.stdout[-3000:])
print("STDERR (마지막 3000자):", result.stderr[-3000:])

# 결과 확인 (.lab 파일 + midi 파일이 나오는 게 정상, README 기준)
print("\n=== 결과 파일 확인 ===")
if os.path.exists(SAVE_DIR):
    for f in os.listdir(SAVE_DIR):
        print(" ", f)

# .lab 파일 내용 미리보기 (코드 시퀀스가 어떤 포맷으로 나오는지 확인)
print("\n=== .lab 파일 내용 미리보기 ===")
for f in os.listdir(SAVE_DIR) if os.path.exists(SAVE_DIR) else []:
    if f.endswith(".lab"):
        lab_path = os.path.join(SAVE_DIR, f)
        with open(lab_path) as fh:
            lines = fh.readlines()
        print(f"--- {f} (앞 10줄) ---")
        for line in lines[:10]:
            print(" ", line.strip())

=== 원곡 존재 확인 ===
/content/drive/MyDrive/stem-remix-assistant/outputs/draft_0.wav 존재? True
복사 완료: /content/BTC-ISMIR19/my_test_audio/draft_0.wav

=== BTC-ISMIR2019 추론 실행 ===
returncode: 1
STDOUT (마지막 3000자): 
STDERR (마지막 3000자): Traceback (most recent call last):
  File "/content/BTC-ISMIR19/test.py", line 22, in <module>
    config = HParams.load("run_config.yaml")
  File "/content/BTC-ISMIR19/utils/hparams.py", line 29, in load
    return cls(**yaml.load(f))
                 ~~~~~~~~~^^^
TypeError: load() missing 1 required positional argument: 'Loader'


=== 결과 파일 확인 ===

=== .lab 파일 내용 미리보기 ===


In [32]:
# =========================================================
# BTC-ISMIR2019 패치: yaml.load()에 Loader 인자 누락 문제 해결
# - PyYAML 5.1+ 부터 yaml.load(f)는 Loader 인자 필수
# - BTC-ISMIR2019는 2019년 코드라 구버전 PyYAML 기준으로 작성됨
# - madmom과 달리 "문법 자체가 제거된" 문제가 아니라
#   "인자 하나가 필수가 된" API 변경이라 한 줄만 고치면 해결됨
# =========================================================

import os

BTC_DIR = "/content/BTC-ISMIR19"
hparams_path = os.path.join(BTC_DIR, "utils", "hparams.py")

print("=== 패치 전 원본 확인 ===")
with open(hparams_path) as f:
    content = f.read()

target = "yaml.load(f)"
if target in content:
    print(f"  '{target}' 발견 -> 패치 진행")
    new_content = content.replace(
        "yaml.load(f)",
        "yaml.load(f, Loader=yaml.FullLoader)"
    )
    with open(hparams_path, "w") as f:
        f.write(new_content)
    print("  패치 완료: yaml.load(f) -> yaml.load(f, Loader=yaml.FullLoader)")
else:
    print(f"  '{target}' 패턴을 찾지 못함 -> hparams.py 내용을 직접 확인 필요")
    print(content[:1000])

print("\n=== 패치 후 확인 ===")
with open(hparams_path) as f:
    print(f.read()[:1500])

=== 패치 전 원본 확인 ===
  'yaml.load(f)' 발견 -> 패치 진행
  패치 완료: yaml.load(f) -> yaml.load(f, Loader=yaml.FullLoader)

=== 패치 후 확인 ===
import yaml


# TODO: add function should be changed
class HParams(object):
    # Hyperparameter class using yaml
    def __init__(self, **kwargs):
        self.__dict__ = kwargs

    def add(self, **kwargs):
        # change is needed - if key is existed, do not update.
        self.__dict__.update(kwargs)

    def update(self, **kwargs):
        self.__dict__.update(kwargs)
        return self

    def save(self, path):
        with open(path, 'w') as f:
            yaml.dump(self.__dict__, f)
        return self

    def __repr__(self):
        return '\nHyperparameters:\n' + '\n'.join([' {}={}'.format(k, v) for k, v in self.__dict__.items()])

    @classmethod
    def load(cls, path):
        with open(path, 'r') as f:
            return cls(**yaml.load(f, Loader=yaml.FullLoader))


if __name__ == '__main__':
    hparams = HParams.load('hparams.yaml')
    p

In [33]:
# =========================================================
# BTC-ISMIR2019 추론 재시도 (yaml 패치 이후)
# - patch_btc_ismir_yaml_loader.py 실행 후 사용
# =========================================================

import subprocess, os

BTC_DIR = "/content/BTC-ISMIR19"
TEST_AUDIO_DIR = os.path.join(BTC_DIR, "my_test_audio")
SAVE_DIR = os.path.join(BTC_DIR, "my_test_result")

print("=== BTC-ISMIR2019 추론 재실행 ===")
result = subprocess.run(
    ["python", "test.py",
     "--audio_dir", TEST_AUDIO_DIR,
     "--save_dir", SAVE_DIR,
     "--voca", "False"],
    cwd=BTC_DIR,
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT (마지막 3000자):", result.stdout[-3000:])
print("STDERR (마지막 3000자):", result.stderr[-3000:])

print("\n=== 결과 파일 확인 ===")
if os.path.exists(SAVE_DIR):
    for f in os.listdir(SAVE_DIR):
        print(" ", f)

print("\n=== .lab 파일 내용 미리보기 ===")
for f in os.listdir(SAVE_DIR) if os.path.exists(SAVE_DIR) else []:
    if f.endswith(".lab"):
        lab_path = os.path.join(SAVE_DIR, f)
        with open(lab_path) as fh:
            lines = fh.readlines()
        print(f"--- {f} (앞 10줄) ---")
        for line in lines[:10]:
            print(" ", line.strip())

=== BTC-ISMIR2019 추론 재실행 ===
returncode: 1
STDOUT (마지막 3000자): 
STDERR (마지막 3000자): I BTC-ISMIR19 09-21 08:49:21.191 test.py:33] label type: Major and minor
Traceback (most recent call last):
  File "/content/BTC-ISMIR19/test.py", line 35, in <module>
    model = BTC_model(config=config.model).to(device)
            ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/content/BTC-ISMIR19/btc_model.py", line 158, in __init__
    self.self_attn_layers = bi_directional_self_attention_layers(*params)
                            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/content/BTC-ISMIR19/btc_model.py", line 106, in __init__
    self.timing_signal = _gen_timing_signal(max_length, hidden_size)
                         ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/BTC-ISMIR19/utils/transformer_modules.py", line 30, in _gen_timing_signal
    np.arange(num_timescales).astype(np.float) * -log_timescale_increment)
                                     ^^^^^^^^
  File "/usr/local/li

In [34]:
# =========================================================
# BTC-ISMIR2019 전체에서 numpy 제거된 별칭(np.float, np.int 등) 스캔
# - madmom 때 grep -E의 lookahead 미지원으로 오탐(0건)이 났던 전례가 있으므로
#   반드시 Python re 모듈로 직접 스캔한다 (grep 안 씀)
# - collections.MutableSequence류 구식 문법도 함께 확인 (madmom에서 먼저
#   겪었던 문제라 여기도 있을 가능성을 배제하지 않고 같이 스캔)
# =========================================================

import os, re

BTC_DIR = "/content/BTC-ISMIR19"

deprecated_pattern = re.compile(r"\bnp\.(float|int|bool|object|str|complex|long)\b(?!\w)")
collections_pattern = re.compile(
    r"from collections import .*(MutableSequence|MutableMapping|MutableSet|Sequence|Mapping|Set|Iterable|Callable)"
)

np_hits = []
collections_hits = []

for root, dirs, files in os.walk(BTC_DIR):
    dirs[:] = [d for d in dirs if d != ".git"]
    for fname in files:
        if not fname.endswith(".py"):
            continue
        fpath = os.path.join(root, fname)
        try:
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                for lineno, line in enumerate(f, start=1):
                    if deprecated_pattern.search(line):
                        np_hits.append((fpath, lineno, line.strip()))
                    if collections_pattern.search(line):
                        collections_hits.append((fpath, lineno, line.strip()))
        except Exception as e:
            print(f"  읽기 실패: {fpath} ({e})")

print(f"=== np.float 등 제거된 numpy 별칭: {len(np_hits)}건 ===")
for fpath, lineno, content in np_hits:
    print(f"  {fpath}:{lineno}: {content}")

print(f"\n=== collections 직접 import (abc 아님): {len(collections_hits)}건 ===")
for fpath, lineno, content in collections_hits:
    print(f"  {fpath}:{lineno}: {content}")

print(f"\n=== 최종 요약 ===")
print(f"np.float류 문제: {len(np_hits)}건 / {len(set(h[0] for h in np_hits))}개 파일")
print(f"collections 문제: {len(collections_hits)}건 / {len(set(h[0] for h in collections_hits))}개 파일")

=== np.float 등 제거된 numpy 별칭: 12건 ===
  /content/BTC-ISMIR19/audio_dataset.py:209: diff = np.diff(chord, axis=0).astype(np.bool)
  /content/BTC-ISMIR19/utils/chords.py:37: CHORD_DTYPE = [('root', np.int),
  /content/BTC-ISMIR19/utils/chords.py:38: ('bass', np.int),
  /content/BTC-ISMIR19/utils/chords.py:39: ('intervals', np.int, (12,)),
  /content/BTC-ISMIR19/utils/chords.py:40: ('is_major',np.bool)]
  /content/BTC-ISMIR19/utils/chords.py:42: CHORD_ANN_DTYPE = [('start', np.float),
  /content/BTC-ISMIR19/utils/chords.py:43: ('end', np.float),
  /content/BTC-ISMIR19/utils/chords.py:46: NO_CHORD = (-1, -1, np.zeros(12, dtype=np.int), False)
  /content/BTC-ISMIR19/utils/chords.py:47: UNKNOWN_CHORD = (-1, -1, np.ones(12, dtype=np.int) * -1, False)
  /content/BTC-ISMIR19/utils/chords.py:290: given_pitch_classes = np.zeros(12, dtype=np.int)
  /content/BTC-ISMIR19/utils/chords.py:323: ivs = np.zeros(12, dtype=np.int)
  /content/BTC-ISMIR19/utils/transformer_modules.py:30: np.arange(num_timesca

In [35]:
# =========================================================
# BTC-ISMIR2019: np.float 등 제거된 numpy 별칭 일괄 패치
# - scan_btc_ismir_np_deprecated.py로 규모를 먼저 확인한 뒤 실행할 것
# - np.float -> float, np.int -> int, np.bool -> bool 등으로 치환
#   (numpy 공식 안내: "그냥 float으로 쓰면 됨, 동작 차이 없음")
# =========================================================

import os, re

BTC_DIR = "/content/BTC-ISMIR19"

# np.float(64가 아닌 것만) -> float, np.int -> int 등으로 치환
# 뒤에 알파벳/숫자가 오지 않는 경우만 매칭 (np.float64는 건드리지 않음)
pattern = re.compile(r"\bnp\.(float|int|bool|object|str|complex|long)\b(?!\w)")

patched_files = []

for root, dirs, files in os.walk(BTC_DIR):
    dirs[:] = [d for d in dirs if d != ".git"]
    for fname in files:
        if not fname.endswith(".py"):
            continue
        fpath = os.path.join(root, fname)
        with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
        new_content = pattern.sub(lambda m: m.group(1), content)
        if new_content != content:
            with open(fpath, "w", encoding="utf-8") as f:
                f.write(new_content)
            patched_files.append(fpath)
            print(f"패치 완료: {fpath}")

print(f"\n총 {len(patched_files)}개 파일 패치 완료")

# 재검증: 패치 후 남은 게 있는지 재스캔
print("\n=== 패치 후 재스캔 ===")
remaining = []
for root, dirs, files in os.walk(BTC_DIR):
    dirs[:] = [d for d in dirs if d != ".git"]
    for fname in files:
        if not fname.endswith(".py"):
            continue
        fpath = os.path.join(root, fname)
        with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
            for lineno, line in enumerate(f, start=1):
                if pattern.search(line):
                    remaining.append((fpath, lineno, line.strip()))

if remaining:
    print(f"아직 {len(remaining)}건 남음:")
    for fpath, lineno, content in remaining:
        print(f"  {fpath}:{lineno}: {content}")
else:
    print("남은 문제 없음 - 패치 완료")

패치 완료: /content/BTC-ISMIR19/audio_dataset.py
패치 완료: /content/BTC-ISMIR19/utils/chords.py
패치 완료: /content/BTC-ISMIR19/utils/transformer_modules.py

총 3개 파일 패치 완료

=== 패치 후 재스캔 ===
남은 문제 없음 - 패치 완료


In [36]:
# =========================================================
# BTC-ISMIR2019 추론 재시도 (yaml 패치 이후)
# - patch_btc_ismir_yaml_loader.py 실행 후 사용
# =========================================================

import subprocess, os

BTC_DIR = "/content/BTC-ISMIR19"
TEST_AUDIO_DIR = os.path.join(BTC_DIR, "my_test_audio")
SAVE_DIR = os.path.join(BTC_DIR, "my_test_result")

print("=== BTC-ISMIR2019 추론 재실행 ===")
result = subprocess.run(
    ["python", "test.py",
     "--audio_dir", TEST_AUDIO_DIR,
     "--save_dir", SAVE_DIR,
     "--voca", "False"],
    cwd=BTC_DIR,
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT (마지막 3000자):", result.stdout[-3000:])
print("STDERR (마지막 3000자):", result.stderr[-3000:])

print("\n=== 결과 파일 확인 ===")
if os.path.exists(SAVE_DIR):
    for f in os.listdir(SAVE_DIR):
        print(" ", f)

print("\n=== .lab 파일 내용 미리보기 ===")
for f in os.listdir(SAVE_DIR) if os.path.exists(SAVE_DIR) else []:
    if f.endswith(".lab"):
        lab_path = os.path.join(SAVE_DIR, f)
        with open(lab_path) as fh:
            lines = fh.readlines()
        print(f"--- {f} (앞 10줄) ---")
        for line in lines[:10]:
            print(" ", line.strip())

=== BTC-ISMIR2019 추론 재실행 ===
returncode: 1
STDOUT (마지막 3000자): 
STDERR (마지막 3000자): I BTC-ISMIR19 09-21 08:52:42.166 test.py:33] label type: Major and minor
Traceback (most recent call last):
  File "/content/BTC-ISMIR19/test.py", line 39, in <module>
    checkpoint = torch.load(model_file)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1602, in load
    raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
_pickle.UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the

In [37]:
# =========================================================
# BTC-ISMIR2019 체크포인트 출처 확인
# - torch.load()가 호출됐다는 건 파일이 이미 존재한다는 뜻
# - 1단계에서 확인 못 했던 "체크포인트가 어디서 왔는지" 지금 확인
# =========================================================

import os

BTC_DIR = "/content/BTC-ISMIR19"
model_dir = os.path.join(BTC_DIR, "model")

print("=== model/ 디렉토리 내용 ===")
if os.path.exists(model_dir):
    for f in os.listdir(model_dir):
        fpath = os.path.join(model_dir, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {f} ({size_mb:.2f} MB)")
else:
    print("  model/ 디렉토리 없음")

# test.py에서 실제로 model_file을 어떻게 정하는지 확인
print("\n=== test.py에서 model_file 관련 코드 ===")
test_py = os.path.join(BTC_DIR, "test.py")
with open(test_py) as f:
    lines = f.readlines()
for i, line in enumerate(lines, start=1):
    if "model_file" in line or "ckpt" in line.lower():
        print(f"  {i}: {line.rstrip()}")

=== model/ 디렉토리 내용 ===
  model/ 디렉토리 없음

=== test.py에서 model_file 관련 코드 ===
  27:     model_file = './test/btc_model_large_voca.pt'
  31:     model_file = './test/btc_model.pt'
  38: if os.path.isfile(model_file):
  39:     checkpoint = torch.load(model_file)


In [38]:
# =========================================================
# BTC-ISMIR2019 패치: torch.load()에 weights_only=False 명시
# - PyTorch 2.6부터 기본값이 True로 바뀌면서, 2019년산 체크포인트
#   (구버전 numpy 객체 포함)를 차단함
# - 공식 저장소에서 받은 신뢰 가능한 체크포인트이므로
#   weights_only=False를 명시해서 우회
# =========================================================

import os, re

BTC_DIR = "/content/BTC-ISMIR19"
test_py = os.path.join(BTC_DIR, "test.py")

with open(test_py) as f:
    content = f.read()

print("=== 패치 전 관련 줄 ===")
for line in content.splitlines():
    if "torch.load(" in line:
        print(" ", line.strip())

# torch.load(model_file) -> torch.load(model_file, weights_only=False)
# 이미 weights_only가 명시된 경우는 건드리지 않도록 방어
pattern = re.compile(r"torch\.load\(([^)]*)\)")

def add_weights_only(match):
    args = match.group(1)
    if "weights_only" in args:
        return match.group(0)
    return f"torch.load({args}, weights_only=False)"

new_content = pattern.sub(add_weights_only, content)

if new_content != content:
    with open(test_py, "w") as f:
        f.write(new_content)
    print("\n패치 완료")
else:
    print("\n변경 사항 없음 (이미 patched이거나 패턴 불일치)")

print("\n=== 패치 후 관련 줄 ===")
with open(test_py) as f:
    for line in f:
        if "torch.load(" in line:
            print(" ", line.strip())

=== 패치 전 관련 줄 ===
  checkpoint = torch.load(model_file)

패치 완료

=== 패치 후 관련 줄 ===
  checkpoint = torch.load(model_file, weights_only=False)


In [39]:
# =========================================================
# BTC-ISMIR2019 추론 2차 재시도 (torch.load 패치 이후)
# =========================================================

import subprocess, os

BTC_DIR = "/content/BTC-ISMIR19"
TEST_AUDIO_DIR = os.path.join(BTC_DIR, "my_test_audio")
SAVE_DIR = os.path.join(BTC_DIR, "my_test_result")

print("=== BTC-ISMIR2019 추론 재실행 (2차) ===")
result = subprocess.run(
    ["python", "test.py",
     "--audio_dir", TEST_AUDIO_DIR,
     "--save_dir", SAVE_DIR,
     "--voca", "False"],
    cwd=BTC_DIR,
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT (마지막 3000자):", result.stdout[-3000:])
print("STDERR (마지막 3000자):", result.stderr[-3000:])

print("\n=== 결과 파일 확인 ===")
if os.path.exists(SAVE_DIR):
    for f in os.listdir(SAVE_DIR):
        print(" ", f)

print("\n=== .lab 파일 내용 미리보기 ===")
for f in os.listdir(SAVE_DIR) if os.path.exists(SAVE_DIR) else []:
    if f.endswith(".lab"):
        lab_path = os.path.join(SAVE_DIR, f)
        with open(lab_path) as fh:
            lines = fh.readlines()
        print(f"--- {f} (앞 15줄) ---")
        for line in lines[:15]:
            print(" ", line.strip())

=== BTC-ISMIR2019 추론 재실행 (2차) ===
returncode: 0
STDOUT (마지막 3000자): 
STDERR (마지막 3000자): I BTC-ISMIR19 09-21 08:54:02.647 test.py:33] label type: Major and minor
I BTC-ISMIR19 09-21 08:54:03.071 test.py:43] restore model
I BTC-ISMIR19 09-21 08:54:03.072 test.py:50] ======== 1 of 1 in progress ========
I BTC-ISMIR19 09-21 08:54:24.153 test.py:53] audio file loaded and feature computation success : /content/BTC-ISMIR19/my_test_audio/draft_0.wav
I BTC-ISMIR19 09-21 08:54:24.647 test.py:96] label file saved : /content/BTC-ISMIR19/my_test_result/draft_0.lab


=== 결과 파일 확인 ===
  draft_0.midi
  draft_0.lab

=== .lab 파일 내용 미리보기 ===
--- draft_0.lab (앞 15줄) ---
  0.000 1.667 F
  1.667 2.685 C
  2.685 4.444 D:min
  4.444 5.556 C
  5.556 6.667 D:min
  6.667 6.944 G
  6.944 8.056 C
  8.056 9.259 F
  9.259 9.352 A#
  9.352 10.000 C
  10.000 11.667 A:min
  11.667 12.222 N
  12.222 13.426 C
  13.426 14.815 F
  14.815 15.000 N


In [41]:
from IPython.display import Audio, display

path = "/content/BTC-ISMIR19/my_test_audio/draft_0.wav"  # .wav 오디오 파일 경로여야 함

print(f"재생: {path}")
display(Audio(path))

재생: /content/BTC-ISMIR19/my_test_audio/draft_0.wav


In [43]:
# =========================================================
# Drive 백업 유틸리티
# - Colab 세션(/content/...)은 런타임이 끊기면 전부 사라짐
# - 세션이 끊기거나 재연결돼도 살아남아야 하는 "중요 파일"은
#   Drive의 지정 폴더로 복사해두기 위한 범용 함수
# - 이후로도 "이 파일 백업해줘" 할 때마다 이 함수만 재사용하면 됨
#
# 확인 결과: 폴더 링크
#   https://drive.google.com/drive/folders/1LhhJdPSujm9gZFHXrjw6_iKwOSNYYV-m
# 는 stem-remix-assistant 폴더 자체(outputs 폴더의 상위)임.
# 즉 /content/drive/MyDrive/stem-remix-assistant 와 동일한 폴더.
# 그 바로 아래에 session_backups라는 새 하위 폴더를 만들어 백업 전용으로 사용.
# =========================================================

import os, shutil

BACKUP_ROOT = "/content/drive/MyDrive/stem-remix-assistant/session_backups"

def backup_file(src_path, subfolder=""):
    """
    단일 파일을 Drive 백업 폴더로 복사.
    subfolder를 지정하면 BACKUP_ROOT 하위에 그 이름의 폴더를 만들어 정리.
    """
    if not os.path.exists(src_path):
        print(f"  [건너뜀] 원본 없음: {src_path}")
        return None

    dest_dir = os.path.join(BACKUP_ROOT, subfolder) if subfolder else BACKUP_ROOT
    os.makedirs(dest_dir, exist_ok=True)

    dest_path = os.path.join(dest_dir, os.path.basename(src_path))
    shutil.copy2(src_path, dest_path)

    size_mb = os.path.getsize(dest_path) / (1024 * 1024)
    print(f"  [완료] {src_path} -> {dest_path} ({size_mb:.2f} MB)")
    return dest_path


def backup_folder(src_dir, subfolder=""):
    """
    폴더 전체(하위 구조 포함)를 Drive 백업 폴더로 복사.
    이미 존재하면 내용을 덮어쓰며 병합(dirs_exist_ok=True).
    """
    if not os.path.exists(src_dir):
        print(f"  [건너뜀] 원본 없음: {src_dir}")
        return None

    folder_name = subfolder if subfolder else os.path.basename(src_dir.rstrip("/"))
    dest_dir = os.path.join(BACKUP_ROOT, folder_name)

    shutil.copytree(src_dir, dest_dir, dirs_exist_ok=True)
    print(f"  [완료] {src_dir}/ -> {dest_dir}/ (전체 폴더)")
    return dest_dir


def backup_many(file_list, subfolder=""):
    """여러 파일을 한 번에 백업. file_list: 경로 문자열 리스트."""
    results = []
    for f in file_list:
        results.append(backup_file(f, subfolder=subfolder))
    return results


if __name__ == "__main__":
    print(f"백업 대상 루트: {BACKUP_ROOT}")
    os.makedirs(BACKUP_ROOT, exist_ok=True)
    print("백업 루트 폴더 준비 완료. 이 파일을 import해서 backup_file/backup_folder/backup_many를 사용하세요.")

백업 대상 루트: /content/drive/MyDrive/stem-remix-assistant/session_backups
백업 루트 폴더 준비 완료. 이 파일을 import해서 backup_file/backup_folder/backup_many를 사용하세요.


In [45]:
# =========================================================
# 오늘 세션에서 생성된 중요 파일들을 Drive로 실제 백업 실행
# - 대상: BTC-ISMIR2019 결과물(lab/midi/변환wav) + MusiConGen 생성 오디오
#         + 패치된 BTC-ISMIR19 소스코드 전체
# - backup_to_drive_utils.py를 먼저 실행(또는 같은 셀에 포함)한 뒤 사용
# =========================================================

from backup_to_drive_utils import backup_file, backup_folder, backup_many, BACKUP_ROOT
import os

print(f"=== 백업 시작 -> {BACKUP_ROOT} ===\n")

# 1. BTC-ISMIR2019 결과물 (lab, midi, 변환된 wav)
print("--- 1. BTC-ISMIR2019 결과물 ---")
BTC_DIR = "/content/BTC-ISMIR19"
result_dir = os.path.join(BTC_DIR, "my_test_result")
backup_folder(result_dir, subfolder="btc_ismir_results")

# 2. MusicGen/MusiConGen 생성 오디오
# 이미 Drive 자체 폴더(outputs/musicongen_generated)에 있을 수 있으므로
# 중복이면 자동으로 덮어써짐 (내용 같으면 문제 없음)
print("\n--- 2. MusiConGen 생성 오디오 ---")
musicongen_output_dir = "/content/drive/MyDrive/stem-remix-assistant/outputs/musicongen_generated"
if os.path.exists(musicongen_output_dir):
    print(f"  이미 Drive 안에 있음: {musicongen_output_dir}")
    print("  (session_backups로 별도 복사하지 않음 - 이미 Drive에 영구 저장된 위치이므로 중복 불필요)")
else:
    print(f"  {musicongen_output_dir} 없음 - 건너뜀")

# 3. 패치된 BTC-ISMIR19 소스코드 전체 (재다운로드+재패치 반복을 피하기 위한 백업)
print("\n--- 3. 패치된 BTC-ISMIR19 소스코드 전체 ---")
backup_folder(BTC_DIR, subfolder="BTC-ISMIR19_patched")

print(f"\n=== 백업 완료. 아래 경로에서 확인 가능 ===")
print(BACKUP_ROOT)
print("(Drive 폴더: https://drive.google.com/drive/u/0/folders/1LhhJdPSujm9gZFHXrjw6_iKwOSNYYV-m 하위 stem-remix-assistant/session_backups)")

ModuleNotFoundError: No module named 'backup_to_drive_utils'

In [47]:
# =========================================================
# backup_to_drive_utils import 오류 해결 (빠른 버전)
# - 이전 버전이 /content 전체(Drive 마운트 포함)를 뒤져서 느렸음
# - Drive는 검색 대상에서 제외하고, 업로드 파일이 있을 가능성이
#   높은 몇 곳만 좁혀서 빠르게 확인
# =========================================================

import subprocess, sys, os

print("=== 1. backup_to_drive_utils.py 파일 위치 찾기 (Drive 제외, 빠르게) ===")

# 흔히 업로드되는 위치들만 직접 확인 (전체 탐색 X)
candidate_dirs = ["/content", "/content/sample_data", os.getcwd()]
found_paths = []

for d in candidate_dirs:
    fpath = os.path.join(d, "backup_to_drive_utils.py")
    if os.path.exists(fpath):
        found_paths.append(fpath)

# 그래도 못 찾으면 /content 바로 아래(1단계 깊이)만 빠르게 훑기
# (-maxdepth로 재귀 깊이 제한 -> Drive 마운트 안까지 파고들지 않음)
if not found_paths:
    result = subprocess.run(
        ["find", "/content", "-maxdepth", "2", "-name", "backup_to_drive_utils.py", "-not", "-path", "*/drive/*"],
        capture_output=True, text=True, timeout=10
    )
    found_paths = [p for p in result.stdout.splitlines() if p.strip()]

if found_paths:
    print("찾은 위치:")
    for p in found_paths:
        print(" ", p)
    target_dir = os.path.dirname(found_paths[0])
else:
    print("파일을 못 찾음.")
    print(f"현재 작업 디렉토리({os.getcwd()})와 /content 바로 아래에 없음.")
    print("Colab 왼쪽 파일 아이콘(폴더 모양)을 클릭 -> backup_to_drive_utils.py가")
    print("실제로 업로드되어 있는지, 있다면 어느 경로인지 직접 확인해줘.")
    target_dir = "/content"

print(f"\n=== 2. sys.path에 폴더 추가: {target_dir} ===")
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)
    print("추가 완료")
else:
    print("이미 sys.path에 있음")

print("\n=== 3. import 재시도 ===")
try:
    from backup_to_drive_utils import backup_file, backup_folder, backup_many, BACKUP_ROOT
    print("import 성공! BACKUP_ROOT =", BACKUP_ROOT)
except ModuleNotFoundError as e:
    print("여전히 실패:", e)
    print(f"\n{target_dir} 안의 .py 파일 목록:")
    if os.path.exists(target_dir):
        for f in os.listdir(target_dir):
            if f.endswith(".py"):
                print(" ", f)

=== 1. backup_to_drive_utils.py 파일 위치 찾기 (Drive 제외, 빠르게) ===
파일을 못 찾음.
현재 작업 디렉토리(/content)와 /content 바로 아래에 없음.
Colab 왼쪽 파일 아이콘(폴더 모양)을 클릭 -> backup_to_drive_utils.py가
실제로 업로드되어 있는지, 있다면 어느 경로인지 직접 확인해줘.

=== 2. sys.path에 폴더 추가: /content ===
이미 sys.path에 있음

=== 3. import 재시도 ===
여전히 실패: No module named 'backup_to_drive_utils'

/content 안의 .py 파일 목록:


In [48]:
# =========================================================
# Drive 백업 - 단일 셀 버전 (import 문제 회피)
# - backup_to_drive_utils.py를 따로 import하지 않고,
#   함수 정의 + 실제 백업 실행을 한 셀에 전부 포함
# - Colab 노트북 새 셀에 이 파일 내용을 그대로 붙여넣고 실행하면 됨
# =========================================================

import os, shutil

# -----------------------------------------------------------
# 1. 백업 유틸 함수 정의 (앞으로도 이 셀을 한 번 실행해두면
#    같은 노트북/커널 내에서는 backup_file 등을 계속 재사용 가능)
# -----------------------------------------------------------

BACKUP_ROOT = "/content/drive/MyDrive/stem-remix-assistant/session_backups"

def backup_file(src_path, subfolder=""):
    """단일 파일을 Drive 백업 폴더로 복사."""
    if not os.path.exists(src_path):
        print(f"  [건너뜀] 원본 없음: {src_path}")
        return None
    dest_dir = os.path.join(BACKUP_ROOT, subfolder) if subfolder else BACKUP_ROOT
    os.makedirs(dest_dir, exist_ok=True)
    dest_path = os.path.join(dest_dir, os.path.basename(src_path))
    shutil.copy2(src_path, dest_path)
    size_mb = os.path.getsize(dest_path) / (1024 * 1024)
    print(f"  [완료] {src_path} -> {dest_path} ({size_mb:.2f} MB)")
    return dest_path

def backup_folder(src_dir, subfolder=""):
    """폴더 전체(하위 구조 포함)를 Drive 백업 폴더로 복사. 이미 있으면 병합."""
    if not os.path.exists(src_dir):
        print(f"  [건너뜀] 원본 없음: {src_dir}")
        return None
    folder_name = subfolder if subfolder else os.path.basename(src_dir.rstrip("/"))
    dest_dir = os.path.join(BACKUP_ROOT, folder_name)
    shutil.copytree(src_dir, dest_dir, dirs_exist_ok=True)
    print(f"  [완료] {src_dir}/ -> {dest_dir}/ (전체 폴더)")
    return dest_dir

def backup_many(file_list, subfolder=""):
    """여러 파일을 한 번에 백업."""
    return [backup_file(f, subfolder=subfolder) for f in file_list]


print(f"백업 대상 루트: {BACKUP_ROOT}")
os.makedirs(BACKUP_ROOT, exist_ok=True)
print("함수 정의 완료: backup_file, backup_folder, backup_many 사용 가능\n")

# -----------------------------------------------------------
# 2. 실제 백업 실행 (오늘 만든 중요 파일들)
# -----------------------------------------------------------

print(f"=== 백업 시작 -> {BACKUP_ROOT} ===\n")

BTC_DIR = "/content/BTC-ISMIR19"

print("--- 1. BTC-ISMIR2019 결과물 (lab/midi) ---")
result_dir = os.path.join(BTC_DIR, "my_test_result")
backup_folder(result_dir, subfolder="btc_ismir_results")

print("\n--- 2. MusiConGen 생성 오디오 ---")
musicongen_output_dir = "/content/drive/MyDrive/stem-remix-assistant/outputs/musicongen_generated"
if os.path.exists(musicongen_output_dir):
    print(f"  이미 Drive 안에 있음: {musicongen_output_dir}")
    print("  (session_backups로 별도 복사하지 않음 - 이미 영구 저장된 위치)")
else:
    print(f"  {musicongen_output_dir} 없음 - 건너뜀")

print("\n--- 3. 패치된 BTC-ISMIR19 소스코드 전체 ---")
backup_folder(BTC_DIR, subfolder="BTC-ISMIR19_patched")

print(f"\n=== 백업 완료 ===")
print(f"확인 경로: {BACKUP_ROOT}")

백업 대상 루트: /content/drive/MyDrive/stem-remix-assistant/session_backups
함수 정의 완료: backup_file, backup_folder, backup_many 사용 가능

=== 백업 시작 -> /content/drive/MyDrive/stem-remix-assistant/session_backups ===

--- 1. BTC-ISMIR2019 결과물 (lab/midi) ---
  [완료] /content/BTC-ISMIR19/my_test_result/ -> /content/drive/MyDrive/stem-remix-assistant/session_backups/btc_ismir_results/ (전체 폴더)

--- 2. MusiConGen 생성 오디오 ---
  이미 Drive 안에 있음: /content/drive/MyDrive/stem-remix-assistant/outputs/musicongen_generated
  (session_backups로 별도 복사하지 않음 - 이미 영구 저장된 위치)

--- 3. 패치된 BTC-ISMIR19 소스코드 전체 ---
  [완료] /content/BTC-ISMIR19/ -> /content/drive/MyDrive/stem-remix-assistant/session_backups/BTC-ISMIR19_patched/ (전체 폴더)

=== 백업 완료 ===
확인 경로: /content/drive/MyDrive/stem-remix-assistant/session_backups


In [42]:
# =========================================================
# BTC-ISMIR2019가 생성한 draft_0.midi를 실제 소리로 들어보기
# - MIDI는 오디오 파형이 아니라 "악보 지시 데이터"라서
#   Audio()에 바로 넣으면 재생되지 않음
# - fluidsynth(무료 사운드폰트 신디사이저)로 MIDI -> WAV 렌더링 후 재생
# =========================================================

import subprocess, os

# 1. fluidsynth + 기본 사운드폰트 설치 (Colab, 최초 1회만 필요)
print("=== fluidsynth 설치 ===")
result = subprocess.run(
    ["apt-get", "install", "-y", "fluidsynth"],
    capture_output=True, text=True
)
print("returncode:", result.returncode)

result = subprocess.run(
    ["pip", "install", "midi2audio", "-q"],
    capture_output=True, text=True
)
print("midi2audio 설치 returncode:", result.returncode)

# 2. 무료 사운드폰트(FluidR3_GM) 다운로드 (없으면)
soundfont_path = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
if not os.path.exists(soundfont_path):
    # apt로 fluid-soundfont-gm 설치 시도
    subprocess.run(["apt-get", "install", "-y", "fluid-soundfont-gm"], capture_output=True, text=True)

print("사운드폰트 존재?", os.path.exists(soundfont_path))

# 3. MIDI -> WAV 변환
from midi2audio import FluidSynth

BTC_DIR = "/content/BTC-ISMIR19"
midi_path = os.path.join(BTC_DIR, "my_test_result", "draft_0.midi")
wav_path = os.path.join(BTC_DIR, "my_test_result", "draft_0_from_midi.wav")

print("\n=== MIDI -> WAV 변환 ===")
print("입력:", midi_path, "존재?", os.path.exists(midi_path))

fs = FluidSynth(sound_font=soundfont_path) if os.path.exists(soundfont_path) else FluidSynth()
fs.midi_to_audio(midi_path, wav_path)

print("출력:", wav_path, "생성됨?", os.path.exists(wav_path))

# 4. 재생
from IPython.display import Audio, display

print(f"\n재생: {wav_path}")
display(Audio(wav_path))

=== fluidsynth 설치 ===
returncode: 100
midi2audio 설치 returncode: 0
사운드폰트 존재? True

=== MIDI -> WAV 변환 ===
입력: /content/BTC-ISMIR19/my_test_result/draft_0.midi 존재? True


FileNotFoundError: [Errno 2] No such file or directory: 'fluidsynth'

In [49]:
# =========================================================
# draft_0.midi -> WAV 변환 재시도 (v2)
# - 이전 시도에서 apt-get install fluidsynth가 returncode 100으로 실패했으나
#   출력을 capture만 하고 화면에 안 띄워서 원인을 알 수 없었음
# - 이번엔 apt-get update 먼저 실행 + 설치 로그를 그대로 출력해서
#   정확한 실패 원인을 확인
# =========================================================

import subprocess, os

print("=== 1. apt-get update (패키지 목록 최신화) ===")
result = subprocess.run(["apt-get", "update"], capture_output=True, text=True)
print("returncode:", result.returncode)
print(result.stdout[-1500:])
print(result.stderr[-1500:])

print("\n=== 2. fluidsynth 설치 (로그 전체 출력) ===")
result = subprocess.run(
    ["apt-get", "install", "-y", "fluidsynth"],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT:\n", result.stdout[-2000:])
print("STDERR:\n", result.stderr[-2000:])

print("\n=== 3. fluidsynth 실행 파일 존재 확인 ===")
which_result = subprocess.run(["which", "fluidsynth"], capture_output=True, text=True)
print("which fluidsynth ->", which_result.stdout.strip() or "(없음)")

# 4. 설치 성공했다면 변환 + 재생까지 진행
if which_result.stdout.strip():
    print("\n=== 4. MIDI -> WAV 변환 ===")
    result = subprocess.run(["pip", "install", "midi2audio", "-q"], capture_output=True, text=True)

    soundfont_path = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
    print("사운드폰트 존재?", os.path.exists(soundfont_path))

    from midi2audio import FluidSynth

    BTC_DIR = "/content/BTC-ISMIR19"
    midi_path = os.path.join(BTC_DIR, "my_test_result", "draft_0.midi")
    wav_path = os.path.join(BTC_DIR, "my_test_result", "draft_0_from_midi.wav")

    print("입력:", midi_path, "존재?", os.path.exists(midi_path))

    fs = FluidSynth(sound_font=soundfont_path) if os.path.exists(soundfont_path) else FluidSynth()
    fs.midi_to_audio(midi_path, wav_path)

    print("출력:", wav_path, "생성됨?", os.path.exists(wav_path))

    from IPython.display import Audio, display
    print(f"\n재생: {wav_path}")
    display(Audio(wav_path))
else:
    print("\nfluidsynth 설치 실패 -> 위 STDERR 로그를 확인해서 원인 파악 필요")
    print("(흔한 원인: apt 소스 목록 문제, 디스크 용량 부족, 권한 문제 등)")

=== 1. apt-get update (패키지 목록 최신화) ===
returncode: 0
//ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:10 https://cli.github.com/packages stable/main amd64 Packages [359 B]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,580 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Packages [56.2 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [1,960 kB]
Get:14 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,149 kB]
Get:15 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:16 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,872 kB]
Get:17 http://archive.ubuntu.com/ubuntu noble-backports/universe amd64 Packages [36.0 kB]
Get:18 http://archive.ubuntu.com/ubuntu noble-backports/main amd64 Packages [49.0 kB]
Get:19 https://cloud.

## 5. MusiConGen 조건화 입력 준비

madmom이 추출한 코드 라벨(`C:maj`, `A:min` 등)을 MusiConGen의 `generate_with_chords_and_beats` 입력 형식(공백으로 구분된 코드 시퀀스 문자열)으로 변환.

**주의**: MusiConGen 공식 스크립트(`generate_chord_beat.py`)에는 `seed` 파라미터가 없음 — 재현성 확보를 위해 아래처럼 `torch.manual_seed()`를 생성 직전에 직접 호출.

In [ ]:
def chords_to_musicongen_format(chord_events, n_bars=8):
    # (start, end, label) 리스트를 MusiConGen 예제 형식("C G A:min F")에 맞춰
    # 마디 단위로 대표 코드만 뽑아 공백 구분 문자열로 변환.
    # 실제 마디 길이(원곡 BPM/박자 기준)에 맞춰 보정 필요 — 여기서는 단순 등간격 샘플링.
    labels = [c[2] for c in chord_events if c[2] != "N"]
    if not labels:
        labels = ["C"]
    step = max(1, len(labels) // n_bars)
    sampled = labels[::step][:n_bars]
    return " ".join(sampled)

chord_str = chords_to_musicongen_format(orig_chords)
bpm_int = int(round(orig_bpm))
print("MusiConGen 조건화 코드 문자열:", chord_str)
print("MusiConGen 조건화 BPM:", bpm_int)


## 6. MusiConGen 로드 및 duration_sec 스윕 생성

In [ ]:
import torch
import audiocraft
from audiocraft.data.audio import audio_write

musicgen = audiocraft.models.MusicGen.get_pretrained("./ckpt/musicongen")

results_dir = "../stem-remix-assistant/experiments/exp_002_musicongen/outputs"
import os
os.makedirs(results_dir, exist_ok=True)

generated_paths = {}

for duration in DURATION_SWEEP:
    torch.manual_seed(SEED)  # 스윕 전 구간 동일 seed 고정

    musicgen.set_generation_params(duration=duration, extend_stride=duration // 2, top_k=250)

    wav = musicgen.generate_with_chords_and_beats(
        [PROMPT],
        [chord_str],
        [bpm_int],
        [4],  # 4/4 박자 가정
    )

    out_path = f"{results_dir}/musicongen_{TARGET_STEM}_{duration}s_seed{SEED}"
    audio_write(out_path, wav[0].cpu(), musicgen.sample_rate, strategy="loudness", loudness_compressor=True)
    generated_paths[duration] = out_path + ".wav"
    print(f"생성 완료: duration={duration}s -> {out_path}.wav")


## 7. 재생성 트랙 Demucs 재분리 → target_stem만 추출

In [ ]:
!pip install demucs -q

import subprocess

demucs_out_dir = f"{results_dir}/demucs_separated"

extracted_stems = {}
for duration, path in generated_paths.items():
    subprocess.run(
        ["python", "-m", "demucs", "-n", "htdemucs", "-o", demucs_out_dir, path],
        check=True,
    )
    track_name = os.path.splitext(os.path.basename(path))[0]
    stem_path = f"{demucs_out_dir}/htdemucs/{track_name}/{TARGET_STEM}.wav"
    extracted_stems[duration] = stem_path
    print(f"duration={duration}s -> {stem_path}")


## 8. Alignment Engine — Beat Align / Transient Align / Time Stretch

재생성 스템을 원곡 스템 기준으로 보정. 세 단계는 독립적으로 나눠서, 어느 단계에서 얼마나 개선되는지 확인 가능하게 구성.

- **Beat Align**: 재생성 스템 첫 비트를 원곡 스템 첫 비트에 맞춰 앞뒤로 자름(오프셋 보정)
- **Time Stretch**: 재생성 스템 BPM과 원곡 BPM의 비율만큼 librosa로 타임 스트레칭
- **Transient Align**: 온셋 단위 미세 보정 (여기서는 온셋 오프셋 표준편차만 측정, 실제 워핑은 다음 단계 과제로 남김)

In [ ]:
import librosa
import soundfile as sf

def beat_align(gen_path, gen_beats, orig_beats, out_path):
    y, sr = librosa.load(gen_path, sr=None)
    if len(gen_beats) == 0 or len(orig_beats) == 0:
        sf.write(out_path, y, sr)
        return out_path
    offset_sec = gen_beats[0] - orig_beats[0]
    offset_samples = int(offset_sec * sr)
    y_aligned = y[max(0, offset_samples):] if offset_samples > 0 else np.concatenate([np.zeros(-offset_samples), y])
    sf.write(out_path, y_aligned, sr)
    return out_path

def time_stretch_to_bpm(path, current_bpm, target_bpm, out_path):
    y, sr = librosa.load(path, sr=None)
    rate = current_bpm / target_bpm if target_bpm else 1.0
    y_stretched = librosa.effects.time_stretch(y, rate=rate)
    sf.write(out_path, y_stretched, sr)
    return out_path

aligned_stems = {}
alignment_dir = f"{results_dir}/aligned"
os.makedirs(alignment_dir, exist_ok=True)

for duration, stem_path in extracted_stems.items():
    gen_bpm = get_bpm(stem_path)
    gen_beats = get_beats(stem_path)

    stretched_path = f"{alignment_dir}/{TARGET_STEM}_{duration}s_stretched.wav"
    time_stretch_to_bpm(stem_path, gen_bpm, orig_bpm, stretched_path)

    gen_beats_stretched = get_beats(stretched_path)
    final_path = f"{alignment_dir}/{TARGET_STEM}_{duration}s_final.wav"
    beat_align(stretched_path, gen_beats_stretched, orig_beats, final_path)

    aligned_stems[duration] = final_path
    print(f"duration={duration}s: 원본 BPM {round(gen_bpm,1)} -> 보정 후 원곡 BPM {round(orig_bpm,1)}에 정렬")


## 9. 정량 지표 계산 (BPM 오차 / Beat alignment / Onset alignment)

`docs/PROPOSAL.md` 4.3절 평가 방법 기준. 세 duration 후보를 비교해 최적값 후보를 정함.

In [ ]:
import pandas as pd

def beat_alignment_score(beats_a, beats_b):
    # 두 비트 시퀀스를 가까운 것끼리 매칭했을 때의 평균 오차(초) — 작을수록 정렬 잘 됨
    if len(beats_a) == 0 or len(beats_b) == 0:
        return None
    errors = [min(abs(a - b) for b in beats_b) for a in beats_a]
    return float(np.mean(errors))

def onset_alignment_score(onsets_a, onsets_b):
    if len(onsets_a) == 0 or len(onsets_b) == 0:
        return None
    errors = [min(abs(a - b) for b in onsets_b) for a in onsets_a]
    return float(np.mean(errors))

rows = []
for duration, final_path in aligned_stems.items():
    final_bpm = get_bpm(final_path)
    final_beats = get_beats(final_path)
    final_onsets = get_onsets(final_path)

    rows.append({
        "duration_sec": duration,
        "bpm_error": round(abs(final_bpm - orig_bpm), 2),
        "beat_alignment_sec": round(beat_alignment_score(final_beats, orig_beats), 3)
            if beat_alignment_score(final_beats, orig_beats) is not None else None,
        "onset_alignment_sec": round(onset_alignment_score(final_onsets, orig_onsets), 3)
            if onset_alignment_score(final_onsets, orig_onsets) is not None else None,
    })

results_df = pd.DataFrame(rows).sort_values("duration_sec")
results_df.to_csv(f"{results_dir}/duration_sweep_results.csv", index=False)
results_df


## 10. 다음 작업

1. 위 표에서 `bpm_error` + `beat_alignment_sec` + `onset_alignment_sec`가 가장 낮은 `duration_sec`를 최적값으로 선택
2. `experiments/exp_002_musicongen/config.yaml`의 `duration_sec`에 확정값 기록, `duration_sec_sweep` 필드는 그대로 이력으로 남김
3. 청취 평가(사람이 직접 듣고 자연스러움 판단) 진행 — `docs/experiments/model_comparison.md`에 기록
4. `experiments/exp_002_musicongen/README.md`의 "결과"·"결론" 섹션 채우기
5. 동일한 duration_sec/seed 스윕 설계를 EXP-003(MusicGen-Melody/Style)에도 적용해 공정 비교 조건 맞추기